In [89]:
import os
print(os.getcwd())

/Users/matt/Desktop/Patent Litigation Analysis/notebooks


In [90]:
#importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re as re

In [101]:
pacer=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/pacer_cases_processed_v4.csv')
cases=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/cases_processed_v4.csv')
names=pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names_cleaned_V2.csv')

In [98]:
names.head()

,case_row_id,case_number,party_row_count,party_type,name_row_count,name,party_type_original,case_number_clean,entity_type
0,1,0:79-cv-06704-JCP,1,Plaintiff,1,Burroghs Wellcome Co.,Plaintiff,0:1979-cv-06704,Organization
1,1,0:79-cv-06704-JCP,2,Defendant,2,Generix Drug Corp.,Defendant,0:1979-cv-06704,Organization
2,3,0:83-cv-06860-JAG,3,Plaintiff,3,Kenneth R. Cornwall,Plaintiff,0:1983-cv-06860,Independent
3,3,0:83-cv-06860-JAG,4,Defendant,4,"U. S. COnstruction Manufacturing, Inc.",Defendant,0:1983-cv-06860,Organization
4,4,0:84-cv-06456-KLR,5,Plaintiff,5,"Monte Carlo Hairpieces, Inc.",Plaintiff,0:1984-cv-06456,Organization


In [99]:
names['party_type'].value_counts()

party_type
Defendant                         217068
Plaintiff                         112988
Counter Claimant                  104947
Counter Defendant                  81737
Consolidated Defendant             14388
Mediator                            3647
Third Party Defendant               3409
Consolidated Plaintiff              3243
Consolidated Counter Claimant       2590
Movant                              2224
Third Party Plaintiff               2128
Intervenor                          1421
Cross Defendant                     1315
Counter Plaintiff                   1270
Cross Claimant                      1069
Consolidated Counter Defendant      1045
Special Master                       899
Miscellaneous                        861
Technical Advisor                    800
Interested Party                     774
Respondent                           361
Notice Only                          280
Amicus                               242
Appellee                             239
Provi

In [100]:
# Drop junk rows
junk_pattern = r'^\d+$|^\d{3}-\d{3}-\d{4}$|^\d{3}\/\d{3}-\d{4}$|^FL\s+\d{5}|^TX\s+\d{5}|^GA\s+\d{5}|^CA\s+\d{5}'
junk_pattern2 = r'^\d+:\d{2}-cv-|^\d{1,2}:\d{2}-cv-|^[\d\s\w]+-ODW-|^\d+ NW \d+ Ave$|Rue Michel|^\d{3,} Hidden Lakes'
names = names[~names['name'].str.contains(junk_pattern, regex=True, na=False)]
names = names[~names['name'].str.contains(junk_pattern2, regex=True, na=False)]
names = names[~names['name'].str.match(r'^\d+-\d+$|^\d+$|^\(.*\)$', na=False)]

drop_values = ['and', 'other', 'inclusive', 'alone', 'son', 'daughter', 'husband', 'wife', 'husband and wife']
names = names[~names['name'].str.strip().str.lower().isin(drop_values)]

def classify_party(name):
    if pd.isna(name):
        return 'Independent'

    unknown_keywords = (
        r'(?i)\bterminated\b|agent\s+of|formerly\s+known\s+as|also\s+known\s+as|'
        r'trading\s+as|now\s+known\s+as|previously\s+named\s+as|incorrectly\s+named\s+as|'
        r'Consolidated\s+(from|Civil\s+Action)|CONSOLIDATED\s+FROM|estate\s+of|on\s+behalf\s+of|'
        r'official\s+capacity|jointly\s+and\s+severally|individually\s+and|'
        r'a\s+sole\s+proprietorship|a\s+natural\s+person|a\s+resident\s+of|'
        r'a\s+California\s+resident|a\s+Florida\s+citizen|\bRelator\b|\bMediator\b|'
        r'Special\s+Master|Technical\s+Advisor|ADR\s+Provider|Liaison\s+Counsel|'
        r'\bDrive\b|\bStreet\b|\bAvenue\b|\bBlvd\b|\bSuite\b|'
        r'MDL\s+Panel|ENE\s+Evaluator|Attorney\s+Settlement\s+Officer|'
        r'an\s+organization\s+form\s+unknown|his\s+marital\s+community|'
        r'formerly\s+doing\s+business\s+as|'
        r'\bexecutor\b|\bregistrar\b|\bexaminer\b|\bmunicipality\b|'
        r'acting\s+Registrar|Esq\.|'
        r'\bAll\s+Defendants\b|\bAll\s+Plaintiffs\b|\bAll\s+defendants\b|'
        r'Seal\s+\d+|Ex\s+Rel'
    )

    corp_keywords = (
        r'(?i)\bInc\.?\b|\bCorp\.?\b|\bCo\.?\b|\bLLC\b|'
        r'\bL\.?\s*L\.?\s*C\.?\b|\bLLP\b|\bLP\b|\bL\.?\s*P\.?\b|\bL\s+P\b|'
        r'\bL\.?\s*C\.?\b|'
        r'\bLtd\.?\b|\bLimited\b|\bIncorporated\b|\bCompany\b|'
        r'\bCorporation\b|\bGmbH\b|\bAG\b|\bS\.?\s*A\.?\b|\bSA\b|'
        r'\bAB\b|\bB\.?\s*V\.?\b|\bN\.?\s*V\.?\b|\bPLC\b|\bPartnership\b|\bPartners\b|'
        r'\bNV\b|\bBV\b|\bA\/S\b|'
        r'\bThe\b|\bUSA\b|\bAmerica\b|'
        r'\bLaboratories\b|\bLabs\b|\bInternational\b|\bBank\b|\bBancorp\b|\bUnion\b|'
        r'\bAssociation\b|\bFoundation\b|\bSolutions\b|\bServices\b|'
        r'\bManufacturing\b|\bEnterprises\b|\bElectronics\b|\bAssociates\b|'
        r'\bN\.?\s*A\.?\b|\bUniversity\b|\bTechnology\b|\bTechnologies\b|\bIndustries\b|'
        r'\bProducts\b|\bSystems\b|\bPharmaceuticals\b|\bEntity\b|\bGroup\b|\bTrust\b|'
        r'\bCommunications\b|\bFitness\b|'
        r'\bAktiebolaget\b|\bKabushiki\s*Kaisha\b|\bOy\b|\bSRL\b|\bS\.R\.L\.\b|'
        r'\bKGaA\b|\bMBH\b|\bMbH\b|\bSAS\b|\bFZE\b|\bHolding\b|\bHospital\b|'
        r'\bResearch\b|\bMedia\b|\bMusic\b|\bOnline\b|\bMobile\b|\bPharmaceutical\b|'
        r'\bBioPharma\b|\bBiotech\b|\bGenetics\b|\bScientific\b|\bLogistics\b|'
        r'\bWheels\b|\bDesign\b|'
        r'\bChevrolet\b|\bToyota\b|\bAirways\b|\bHotels\b|\bResorts\b|\bNutrition\b|'
        r'\bFootwear\b|\bKnives\b|\bAlloys\b|\bPlastics\b|\bFurniture\b|\bGifts\b|'
        r'\bPayments\b|\bTelematics\b|\bBiosystems\b|\bHealthcare\b|\bHolster\b|'
        r'\bArchery\b|\bSporting\s+Goods\b|\bContracting\b|\bEngineering\b|\bTelecom\b|\bLighting\b|'
        r'\bd\/b\/a\b|\bdba\b|doing\s+business\s+as'
    )

    as_end = r'\bAS,?\s*$|\bAs,?\s*$'

    if re.search(corp_keywords, name) or re.search(as_end, name):
        return 'Corporation'
    elif re.search(unknown_keywords, name):
        return 'Unknown'
    else:
        return 'Independent'

names['entity_type'] = names['name'].apply(classify_party)

specific_corps = [
    'SANOFI-AVENTIS', 'Sanofi-Aventis', 'SANOFI', 'Sanofi',
    'AKTIEBOLAGET HASSLE', 'Aktiebolaget Hassle',
    'Verizon', 'VERIZON', 'Verizon Communications', 'Verizon Communications Inc.',
    'Verizon Wireless', 'Verizon Business', 'Verizon Services Corp.',
    'Verizon Enterprise Solutions', 'Verizon Data Services',
    'Verizon Online LLC', 'Verizon Internet Services Inc.',
    'Johnson & Johnson', 'Johnson and Johnson', 'JOHNSON & JOHNSON',
    'Johnson & Johnson Services, Inc.', 'Johnson & Johnson Consumer Inc.',
    'Johnson & Johnson Health Care Systems Inc.',
    'Johnson & Johnson Pharmaceutical Research and Development, L.L.C.',
    'Johnson & Johnson Vision Care, Inc.',
    'Johnson & Johnson Medical Devices Companies',
    'Johnson & Johnson Medical Devices Companies, Inc.',
    'Apple', 'APPLE', 'Apple Inc.', 'Apple Computer, Inc.',
    'Apple Computer Inc.', 'Apple Incorporated', 'Apple Corporation',
    'Apple Corporation Inc.',
    'USPTO', 'Commissioner of Patents and Trademarks',
    'COMMISSIONER OF PATENTS AND TRADEMARKS',
    'United States Patent and Trademark Office',
    'UNITED STATES PATENT AND TRADEMARK OFFICE',
    'Wyeth', 'WYETH',
    'GlaxoSmithKline', 'GLAXOSMITHKLINE',
    'Morgan Stanley', 'MORGAN STANLEY',
    'Microsoft', 'MICROSOFT',
    'Amazon.com', 'Amazon', 'AMAZON',
    'Merck & Cie', 'Merck KGaA', 'MERCK',
    'Myriad Genetics', 'MYRIAD GENETICS',
    'British Telecommunications', 'BRITISH TELECOMMUNICATIONS',
    'Thomson Licensing', 'THOMSON LICENSING',
    'Disney Online', 'DISNEY ONLINE',
    'Sony BMG Music Entertainment', 'SONY BMG MUSIC ENTERTAINMENT',
    'Neurografix', 'NeuroGrafix', 'NEUROGRAFIX',
    'Hydro-Quebec', 'HYDRO-QUEBEC',
    'Zions Bancorporation', 'ZIONS BANCORPORATION',
    'KeyCorp', 'KEYCORP',
    'IAC/InterActiveCorp', 'IAC/INTERACTIVECORP',
    'Sun Pharma Global FZE', 'SUN PHARMA GLOBAL FZE',
    'Cephalon France', 'CEPHALON FRANCE',
    'Polar Electro Oy', 'POLAR ELECTRO OY',
    'Stratagene', 'STRATAGENE',
    'Astra Aktiebolag', 'ASTRA AKTIEBOLAG',
    'Commonwealth Scientific and Industrial Research Organisation',
    'BMG Music', 'BMG MUSIC',
    'Stanley Works', 'STANLEY WORKS',
    'Norton-Alcoa Proppants',
    'GODECKE AKTIENGESELLSCHAFT', 'Godecke Aktiengesellschaft',
    'JANSSEN R&D IRELAND', 'JANSSEN SCIENCES IRELAND UC',
    'MERIAL SAS', 'Zydus Cadila', 'SCR Pharmatop',
    'Manhattan Design Studio', 'American Covers', 'City of Hope',
    'New York Medical College',
    'Shionogi Seiyaku Kabushiki Kaisha',
    'Telefonaktiebolaget LM Ericsson', 'Telefonaktiebolaget LM Ericcson',
    'Microsoft Mobile Oy', 'Basic Research',
    "Fournier Industrie et Sante'", 'PF Prism C.V.',
    'Hotels.com', "Sam's Club",
    'ConocoPhillips', 'Alcatel-Lucent', 'Oakley', 'Wal-Mart Stores', 'DirecTV',
    'Home Depot', '3M', 'Cartier', 'Atlas Copco', 'Astellas Pharma US',
    'Synchrony Financial', 'FreshBooks', 'Honeywell Scanning & Mobility',
    'ANDREWS MCMEEL UNIVERSAL', 'PE Applied Biosystems Division',
    'Phillips Petroleum', 'JetBlue Airways Coporation',
    'BASF Aktiengesellschaft', 'Neste Oil Oyj',
    'Merck Patent Gesellschaft mit beschrankter Haftung',
    'Pechiney Emballage Flexible Europe', 'Industrias Auxiliares Faus S.L.',
    'Aliphcom', 'AliphCom', 'Endorecherche', 'Cellectis bioresearch',
    'Ivera Medical', 'Laserscope', 'Pacific Coast Lighting',
    'Center for Neurologic Study', 'Laser Eye Center',
    'President and Fellows of Harvard College',
    'ORCHID HEALTHCARE', 'Desantis Holster & Leather Goods',
    'Kershaw Knives', 'Browning', 'Xlear',
    'Turn-Key-Tech', 'Tinkers & Chance', 'Dots of Fun', 'Centre One',
    'Denmel Holdings', 'InNova', 'Planet Blue', 'Etagz', 'SkipPrint',
    'CpuMate', 'Zoomers', 'ABC Companies', "Santa's Best",
    'Southern Exposure Seed Exchange', '3Form', 'Waterway Plastics',
    'Mobile Hi-Tech Wheels', 'Alba Performance Wheels',
    'Golden Trade, S.R.L.', 'DBI/SALA', 'Teva Canada Innovation GP-SENC',
    'DMSI', 'CaseCrown', 'SB Tactical', 'MAGIXHOSE', 'ID TECH',
    'AnywhereCommerce', 'GoInterpay', 'Connected Telematics',
    'Pointer Footwear', 'Iherb.com', 'Europerfumes', 'Clearwater Toyota',
    'Country Club Software', 'Boca Dental Implant Center',
    'Bean Station Furniture', 'Remco Toys', 'Dobler Chevrolet',
    'Colony Hotels & Resorts', 'N.S. Bienstock Agency',
    'R.A. Brown & Assoc.', 'Idea Source', 'Gilat-To-Home',
    'A2Z Development Center', 'Heartland MicroPayments',
    'Missoula Urban Transportation District', 'Brachysciences',
    'MHT Luxury Alloys', 'Strong Idea', 'CREATIVE BRICK AND CONCRETE',
    'Axis Global Logistics', 'eWonderworld', 'Proxcardpro.com',
    'Siskiyou Gifts', 'McRo Infc', 'Maicroaire Surgical Instruments',
    'Spectronics Corporat', 'Robinair Division',
    'Target Stores', 'Radio Shack', 'Cepheid', 'Catheter Connections',
    'Springlite', 'California Air Resources Board',
    'Transportation Security Administration', 'United States Army',
    'Transamerican Auto Parts', 'Express Metals Recycling', 'UV-Nails',
    'Mamiyaleaf', 'Fish & Richardson, P.C.', 'Feldman Gale, PA',
    'Mayback & Hoffman, PA', 'Mayback & Hoffman, P.A.',
    'South Florida nuclear Medicine, PA', 'Nuclear Medicine Scanning, PA',
    'Jack T. Krauser, D.M.D., P.A.', 'Jack T. Krauser, D.M.D., PLLC',
    'Shell Oil', 'Pelco', 'Kransco', 'Travelon', 'Larson Archery',
    'Paragon Sporting Goods', "Toys 'R' Us", "Toys \"R\" Us",
    'Asus Computer Intl', 'UCB Societe Anonyme', 'INDUSTRIE NATUZZI S.P.A.',
    'Scintilla, A.G.', 'Anstalt Pulsair', 'Kouvato', 'Kuvato',
    'MW Contracting', 'MW Engineering', 'AH Lighting', 'Diode Led',
    'Integra Telecom', 'Cellular Factory Wholesale', 'Janakal Tool and Supply',
    'Fantasy Game Unlimited', 'Hill Country Custom Cycles',
    'Skywaytools.com', 'Matts Tools', 'VoiceFill', 'West Direct',
    'Promounts', 'Kristens Kloset', 'ePromoLogo', 'Prima Force',
    'Go Rhino', 'Vivitar', 'Star Trac Health & Firness',
    'Caso Germany', 'Toys To Grow On', 'C. Orrico Boutique',
    'Max.com', 'Edizone', 'Brenthaven', 'Luna-Park', 'Celebrity',
    'Goody', 'IGIT',
    'St. Jude Medical', 'NYSE Euronext', 'Cellectis Bioresearch',
    'Trademark Lending', 'P2F Holdings', 'Cavalry Empire',
    'Southern RobotX', 'Brinks Hofer Gilson & Lione, PC',
    'Kusner & Jaffe', 'Mueting, Raasch & Gebhardt, PA',
    'NYCTM', 'Rist and Storm Catcher', 'Adidas Defendants',
    'Realtime Data, LLC D/B/A IXO',
    'Verizon Communications Inc. d/b/a Verizon Wireless',
    'Rembrandt Technologies, LLC d/b/a Remstream',
    'Rembrandt Technologies, LLC d/b/a Remstream Solutions',
    'Rembrandt Technologies, LLC d/b/a Remstream Services',
    'also known asKlausner Technologies, Inc.',
    'National Oilwell Varco, LP dba NOV Brandt and dba Brandt',
    'formerly known as LVL Patent Goup, LLC',
    'also known asREALTIME DATA, LLC D/B/A/ IXO',
]
names.loc[names['name'].isin(specific_corps), 'entity_type'] = 'Corporation'

names.loc[
    (names['entity_type'] == 'Independent') &
    (names['name'].str.match(r'^[A-Z]{2,3}$', na=False)),
    'entity_type'
] = 'Corporation'

print(names['entity_type'].value_counts())

entity_type
Corporation    423986
Independent     75292
Unknown         60454
Name: count, dtype: int64


In [94]:
junk_unknowns = [
    '99 S. Almaden Blvd. #600',
    '3161 Michelson Drive, Suite 800',
    '4616 W. Howard Lane, Suite 960',
    '2101 Middle River Drive',
    '500 W. Cypress Creek Road, Suite 350',
    '111 Eighth Avenue',
    'Suite 380',
]
names = names[~names['name'].isin(junk_unknowns)]

names.loc[names['name'].isin([
    'an individualalso known asOscar Liao',
    'an iindividualalso known asChien Wan Tze Liao',
    'an individualalso known asBen Lai',
    'an individualalso known as"Martin Wang"',
]), 'entity_type'] = 'Independent'

print(names['entity_type'].value_counts())

entity_type
Corporation    423986
Independent     75296
Unknown         60450
Name: count, dtype: int64


In [95]:
names['entity_type'] = names['entity_type'].str.replace('Corporation', 'Organization')

In [62]:
individual_mask = names['name'].str.contains(r'\ban individual\b|\bDoes\b', case=False, na=False)
names.loc[individual_mask, 'entity_type'] = 'Independent'
names.loc[individual_mask, 'standardized_org_name'] = None

org_mask = names['entity_type'] == 'Organization'
names.loc[org_mask, 'standardized_org_name'] = names.loc[org_mask, 'name'].apply(standardize_org_name)

names.loc[org_mask, 'standardized_org_name'] = names.loc[org_mask, 'standardized_org_name'].apply(get_parent_org)

In [63]:
import re

SUFFIX_MAP = [
    (r',?\s+Incorporated\.?$',  'Inc.'),
    (r',?\s+Inc[,\.]?$',        'Inc.'),
    (r',?\s+Corporation\.?$',   'Corp.'),
    (r',?\s+Corp[,\.]?$',       'Corp.'),
    (r',?\s+LLC\.?$',           'LLC'),
    (r',?\s+L\.L\.C\.?$',       'LLC'),
    (r',?\s+Limited\.?$',       'Ltd.'),
    (r',?\s+Ltd[,\.]?$',        'Ltd.'),
    (r',?\s+L\.P\.?$',          'L.P.'),
    (r',?\s+Company\.?$',       'Co.'),
    (r',?\s+Co[,\.]?$',         'Co.'),
]

def standardize_org_name(name: str) -> str:
    if not isinstance(name, str):
        return name

    name = name.strip()

    if name == name.upper():
        name = name.title()

    for pattern, replacement in SUFFIX_MAP:
        new_name = re.sub(pattern, ', ' + replacement, name, flags=re.IGNORECASE)
        if new_name != name:
            name = new_name
            break

    return name

org_mask = names['entity_type'] == 'Organization'
names.loc[org_mask, 'standardized_org_name'] = names.loc[org_mask, 'name'].apply(standardize_org_name)

In [65]:
# Parent company map

PARENT_MAP = {

    "Apple, Inc.":                          "Apple Inc.",
    "Apple, Corp.":                         "Apple Inc.",
    "Apple Computer, Inc.":                 "Apple Inc.",
    "Apple Computer Company, Inc.":         "Apple Inc.",
    "Apple Computers, Inc.":                "Apple Inc.",
    "Apple Computer inc":                   "Apple Inc.",
    "Apple Incorporated":                   "Apple Inc.",
    "Apple Inc,":                           "Apple Inc.",
    "APPLE INC.":                           "Apple Inc.",
    "APPLE, INC.":                          "Apple Inc.",
    "Apple Inc":                            "Apple Inc.",

    "Samsung Electronics America, Inc.":            "Samsung Electronics",
    "Samsung Electronics Co., Ltd.":                "Samsung Electronics",
    "Samsung Telecommunications America, LLC":      "Samsung Electronics",
    "Samsung Semiconductor, Inc.":                  "Samsung Electronics",
    "Samsung Electronics Co, Ltd.":                 "Samsung Electronics",
    "Samsung TeleCommunications America, LLC":       "Samsung Electronics",
    "Samsung Electronics Co., LTD.,":               "Samsung Electronics",
    "Samsung Austin Semiconductor, LLC":            "Samsung Electronics",
    "Samsung Telecommunications America LLP":       "Samsung Electronics",
    "Samsung Electronics America, Inc.,":           "Samsung Electronics",
    "Samsung Electronics Company, Ltd.":            "Samsung Electronics",
    "Samsung Electronics, Co., Ltd.":               "Samsung Electronics",
    "Samsung SDI Co., Ltd.":                        "Samsung Electronics",
    "Samsung Opto-Electronics America, Inc.":       "Samsung Electronics",
    "Samsung Electronics USA, Inc.":                "Samsung Electronics",
    "Samsung Techwin Co., Ltd.":                    "Samsung Electronics",
    "Samsung SDI Co, Ltd.":                         "Samsung Electronics",
    "Samsung Telecommunications America, Inc.":     "Samsung Electronics",
    "Samsung America, Inc.":                        "Samsung Electronics",
    "Samsung Electronics America":                  "Samsung Electronics",
    "Samsung Electronics":                          "Samsung Electronics",
    "Samsung, Inc.":                                "Samsung Electronics",
    "Samsung, Corp.":                               "Samsung Electronics",
    "Samsung Electronics, Inc.":                    "Samsung Electronics",
    "Samsung Display Co., Ltd.":                    "Samsung Electronics",
    "Samsung Mobile Display Co., Ltd.":             "Samsung Electronics",
    "Samsung Electro-Mechanics Co., Ltd.":          "Samsung Electronics",
    "Samsung TeleCommunications America, Inc.":     "Samsung Electronics",
    "Samsung Mobile":                               "Samsung Electronics",

    "Sony Electronics, Inc.":                       "Sony Corp.",
    "Sony, Corp.":                                  "Sony Corp.",
    "Sony Corporation of America":                  "Sony Corp.",
    "Sony Ericsson Mobile Communications (USA), Inc.": "Sony Corp.",
    "Sony Computer Entertainment America, Inc.":    "Sony Corp.",
    "Sony Mobile Communications (USA), Inc.":       "Sony Corp.",
    "Sony Computer Entertainment America, LLC":     "Sony Corp.",
    "Sony Computer Entertainment, Inc.":            "Sony Corp.",
    "Sony Mobile Communications AB":                "Sony Corp.",
    "Sony BMG Music Entertainment":                 "Sony Corp.",
    "Sony DADC US, Inc.":                           "Sony Corp.",
    "Sony Online Entertainment, LLC":               "Sony Corp.",
    "Sony Music Entertainment, Inc.":               "Sony Corp.",
    "Sony Pictures Entertainment, Inc.":            "Sony Corp.",
    "Sony Corp of America":                         "Sony Corp.",
    "Sony Corporation Of America":                  "Sony Corp.",
    "Sony Interactive Entertainment, Inc.":         "Sony Corp.",

    "HTC America, Inc.":                    "HTC Corp.",
    "HTC, Corp.":                           "HTC Corp.",
    "HTC (BVI), Corp.":                     "HTC Corp.",
    "HTC BVI, Corp.":                       "HTC Corp.",
    "HTC America, Inc.,":                   "HTC Corp.",
    "HTC (B.V.I.), Corp.":                  "HTC Corp.",
    "HTC America Holding, Inc.":            "HTC Corp.",
    "HTC USA, Inc.":                        "HTC Corp.",
    "HTC America Innovation, Inc.":         "HTC Corp.",
    "HTC, Inc.":                            "HTC Corp.",
    "HTC America":                          "HTC Corp.",
    "HTC America Holdings, Inc.":           "HTC Corp.",

    "LG Electronics, Inc.":                 "LG Electronics",
    "LG Electronics USA, Inc.":             "LG Electronics",
    "LG Electronics U.S.A., Inc.":          "LG Electronics",
    "LG Electronics Mobilecomm U.S.A., Inc.": "LG Electronics",
    "LG Electronics Mobilecomm USA, Inc.":  "LG Electronics",
    "LG Electronics MobileComm U.S.A., Inc.": "LG Electronics",
    "LG Electronics MobileComm USA, Inc.":  "LG Electronics",
    "LG Display Co., Ltd.":                 "LG Electronics",
    "LG Display America, Inc.":             "LG Electronics",
    "LG Electronics U.S.A., Inc.,":         "LG Electronics",
    "LG Electronics":                       "LG Electronics",
    "LG Electronics, U.S.A., Inc.":         "LG Electronics",

    "Microsoft, Corp.":                     "Microsoft Corp.",
    "Microsoft, Corp":                      "Microsoft Corp.",

    "Google, Inc.":                         "Google Inc.",

    "Amazon.com, Inc.":                     "Amazon.com Inc.",

    "Dell, Inc.":                           "Dell Inc.",

    "Intel, Corp.":                         "Intel Corp.",

    "Cisco Systems, Inc.":                  "Cisco Systems Inc.",

    "Qualcomm, Inc.":                       "Qualcomm Inc.",
    "Qualcomm Technologies, Inc.":          "Qualcomm Inc.",
    "Qualcomm Atheros, Inc.":               "Qualcomm Inc.",
    "QUALCOMM, Inc.":                       "Qualcomm Inc.",
    "Qualcomm-Atheros, Inc.":               "Qualcomm Inc.",
    "Qualcomm Innovation Center, Inc.":     "Qualcomm Inc.",

    "Broadcom, Corp.":                      "Broadcom Corp.",
    "BROADCOM, Corp.":                      "Broadcom Corp.",

    "Motorola, Inc.":                       "Motorola Inc.",
    "Motorola Mobility, Inc.":              "Motorola Inc.",
    "Motorola Mobility, LLC":               "Motorola Inc.",
    "Motorola Solutions, Inc.":             "Motorola Inc.",
    "Motorola Mobility Holdings, Inc.":     "Motorola Inc.",
    "Motorola, Corp.":                      "Motorola Inc.",
    "Motorola Mobility Holdings, LLC":      "Motorola Inc.",
    "Motorola Mobility International, Ltd.": "Motorola Inc.",

    "Nokia, Inc.":                          "Nokia Corp.",
    "Nokia, Corp.":                         "Nokia Corp.",
    "Nokia Mobile Phones, Inc.":            "Nokia Corp.",
    "Nokia Holding, Inc.":                  "Nokia Corp.",
    "Nokia Siemens Networks US, LLC":       "Nokia Corp.",
    "Nokia Mobile Phones Americas, Inc.":   "Nokia Corp.",
    "Nokia USA, Inc.":                      "Nokia Corp.",
    "Nokia Mobile Phones, Ltd.":            "Nokia Corp.",
    "Nokia Networks, Inc.":                 "Nokia Corp.",
    "Nokia Products, Corp.":                "Nokia Corp.",
    "Nokia Mobile Phones":                  "Nokia Corp.",

    "BlackBerry, Corp.":                    "BlackBerry Ltd.",
    "BlackBerry, Ltd.":                     "BlackBerry Ltd.",
    "Blackberry, Ltd.":                     "BlackBerry Ltd.",
    "Blackberry, Corp.":                    "BlackBerry Ltd.",
    "Research In Motion, Corp.":            "BlackBerry Ltd.",
    "Research In Motion, Ltd.":             "BlackBerry Ltd.",
    "Research in Motion, Ltd.":             "BlackBerry Ltd.",
    "Research in Motion, Corp.":            "BlackBerry Ltd.",
    "Research In Motion, Inc.":             "BlackBerry Ltd.",

    "Huawei Device USA, Inc.":              "Huawei Technologies",
    "Huawei Technologies USA, Inc.":        "Huawei Technologies",
    "Huawei Technologies Co., Ltd.":        "Huawei Technologies",
    "Huawei Devices USA, Inc.":             "Huawei Technologies",
    "Huawei Technologies Co, Ltd.":         "Huawei Technologies",
    "Huawei Device Co., Ltd.":              "Huawei Technologies",
    "Huawei America, Inc.":                 "Huawei Technologies",
    "Huawei Technologies, Co., Ltd.":       "Huawei Technologies",
    "Huawei Technologies USA":              "Huawei Technologies",

    "Toshiba, Corp.":                       "Toshiba Corp.",
    "Toshiba America Information Systems, Inc.": "Toshiba Corp.",
    "Toshiba America, Inc.":                "Toshiba Corp.",
    "Toshiba America Electronic Components, Inc.": "Toshiba Corp.",
    "Toshiba America Consumer Products, LLC": "Toshiba Corp.",
    "Toshiba America Medical Systems, Inc.": "Toshiba Corp.",
    "Toshiba America Business Solutions, Inc.": "Toshiba Corp.",
    "Toshiba International, Corp.":         "Toshiba Corp.",

    "Panasonic Corporation of North America": "Panasonic Corp.",
    "Panasonic, Corp.":                     "Panasonic Corp.",
    "Panasonic Corp. of North America":     "Panasonic Corp.",
    "Panasonic Corporation Of North America": "Panasonic Corp.",
    "Panasonic Consumer Electronics, Co.":  "Panasonic Corp.",
    "Panasonic Corporation of North America, Inc.": "Panasonic Corp.",
    "Panasonic Corporation of America":     "Panasonic Corp.",

    "AT&T Mobility, LLC":                   "AT&T Inc.",
    "AT&T, Inc.":                           "AT&T Inc.",
    "AT&T, Corp.":                          "AT&T Inc.",
    "AT&T Services, Inc.":                  "AT&T Inc.",
    "AT&T Operations, Inc.":                "AT&T Inc.",
    "AT&T Wireless Services, Inc.":         "AT&T Inc.",
    "AT&T Mobility II, LLC":                "AT&T Inc.",
    "AT&T Internet Services":               "AT&T Inc.",
    "At&T, Corp.":                          "AT&T Inc.",
    "At&T Mobility, LLC":                   "AT&T Inc.",
    "AT&T Mobility, Corp.":                 "AT&T Inc.",

    "Verizon Communications, Inc.":         "Verizon Communications",
    "Cellco Partnership d/b/a Verizon Wireless": "Verizon Communications",
    "Verizon Services, Corp.":              "Verizon Communications",
    "Verizon Wireless":                     "Verizon Communications",
    "Verizon Business Network Services, Inc.": "Verizon Communications",
    "Verizon Online, LLC":                  "Verizon Communications",
    "Verizon Data Services, LLC":           "Verizon Communications",
    "Verizon South, Inc.":                  "Verizon Communications",
    "Verizon Wireless, Inc.":               "Verizon Communications",
    "Verizon Virginia, Inc.":               "Verizon Communications",
    "Verizon California, Inc.":             "Verizon Communications",

    "Sprint Spectrum, L.P.":                "Sprint Corp.",
    "Sprint Nextel, Corp.":                 "Sprint Corp.",
    "Sprint Solutions, Inc.":               "Sprint Corp.",
    "Sprint, Corp.":                        "Sprint Corp.",
    "Sprint Communications Company, L.P.":  "Sprint Corp.",
    "Sprint Communications, Inc.":          "Sprint Corp.",
    "Sprint Spectrum LP":                   "Sprint Corp.",

    "T-Mobile USA, Inc.":                   "T-Mobile USA Inc.",

    "Comcast, Corp.":                       "Comcast Corp.",
    "Comcast Cable Communications, LLC":    "Comcast Corp.",
    "Comcast Interactive Media, LLC":       "Comcast Corp.",

    "Hewlett-Packard, Co.":                 "HP Inc.",
    "Hewlett Packard, Co.":                 "HP Inc.",
    "Hewlett-Packard Development Company, L.P.": "HP Inc.",
    "Hewlett-Packard, Corp.":               "HP Inc.",
    "Hewlett Packard Enterprise, Co.":      "HP Inc.",
    "Hewlett Packard, Corp.":               "HP Inc.",

    "International Business Machines, Corp.": "IBM Corp.",

    "Oracle, Corp.":                        "Oracle Corp.",

    "Adobe Systems, Inc.":                  "Adobe Inc.",

    "Symantec, Corp.":                      "Symantec Corp.",

    "Canon, Inc.":                          "Canon Inc.",
    "Canon U.S.A., Inc.":                   "Canon Inc.",
    "Canon USA, Inc.":                      "Canon Inc.",
    "Canon Solutions America, Inc.":        "Canon Inc.",
    "Canon Computer Systems, Inc.":         "Canon Inc.",

    "Fujitsu, Ltd.":                        "Fujitsu Ltd.",
    "Fujitsu America, Inc.":                "Fujitsu Ltd.",
    "Fujitsu Computer Products of America, Inc.": "Fujitsu Ltd.",
    "Fujitsu Network Communications, Inc.": "Fujitsu Ltd.",
    "Fujitsu Computer Systems, Corp.":      "Fujitsu Ltd.",
    "Fujitsu Microelectronics, Inc.":       "Fujitsu Ltd.",
    "Fujitsu Semiconductor, Ltd.":          "Fujitsu Ltd.",
    "Fujitsu Semiconductor America, Inc.":  "Fujitsu Ltd.",

    "Epson America, Inc.":                  "Seiko Epson Corp.",
    "Seiko Epson, Corp.":                   "Seiko Epson Corp.",

    "Nintendo of America, Inc.":            "Nintendo Co. Ltd.",
    "Nintendo Co., Ltd.":                   "Nintendo Co. Ltd.",
    "Nintendo Co, Ltd.":                    "Nintendo Co. Ltd.",
    "Nintendo Company, Ltd.":               "Nintendo Co. Ltd.",
    "Nintendo Of America, Inc.":            "Nintendo Co. Ltd.",
    "Nintendo of America":                  "Nintendo Co. Ltd.",
    "Nintendo of North America, Inc.":      "Nintendo Co. Ltd.",

    "Acer America, Corp.":                  "Acer Inc.",
    "Acer, Inc.":                           "Acer Inc.",

    "General Electric, Co.":               "General Electric Co.",
    "General Electric Capital, Corp.":     "General Electric Co.",
    "General Electric, Corp.":             "General Electric Co.",
    "The General Electric, Co.":           "General Electric Co.",

    "General Motors, Corp.":               "General Motors Co.",
    "General Motors, LLC":                 "General Motors Co.",
    "General Motors, Co.":                 "General Motors Co.",
    "General Motors Acceptance, Corp.":    "General Motors Co.",
    "General Motors of Canada, Ltd.":      "General Motors Co.",

    "Ford Motor, Co.":                     "Ford Motor Co.",

    "BMW of North America, LLC":           "BMW AG",
    "Bayerische Motoren Werke AG":         "BMW AG",

    "Mercedes-Benz USA, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz U.S. International, Inc.": "Mercedes-Benz Group",
    "Mercedes Benz USA, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz Usa, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz of North America, Inc.": "Mercedes-Benz Group",

    "Nissan North America, Inc.":          "Nissan Motor Co.",
    "Nissan Motor Co., Ltd.":              "Nissan Motor Co.",
    "Nissan Motor Co, Ltd.":               "Nissan Motor Co.",
    "Nissan Motor Company, Ltd.":          "Nissan Motor Co.",
    "Nissan Motor, Co.":                   "Nissan Motor Co.",

    "Hyundai Motor America":               "Hyundai Motor Co.",
    "Hyundai Motor, Co.":                  "Hyundai Motor Co.",
    "Hyundai Motor America, Inc.":         "Hyundai Motor Co.",
    "Hyundai Motor Manufacturing Alabama, LLC": "Hyundai Motor Co.",

    "Wal-Mart Stores, Inc.":               "Walmart Inc.",
    "Wal-Mart Stores Texas, LLC":          "Walmart Inc.",
    "Wal-Mart, Inc.":                      "Walmart Inc.",
    "Wal-Mart Stores East, LP":            "Walmart Inc.",
    "Wal-Mart Stores":                     "Walmart Inc.",

    "Target, Corp.":                       "Target Corp.",

    "Best Buy Co., Inc.":                  "Best Buy Co.",
    "Best Buy Stores, L.P.":               "Best Buy Co.",
    "Best Buy Co, Inc.":                   "Best Buy Co.",
    "Best Buy Company, Inc.":              "Best Buy Co.",
    "Best Buy, Co.":                       "Best Buy Co.",

    "Costco Wholesale, Corp.":             "Costco Wholesale Corp.",

    "Nike, Inc.":                          "Nike Inc.",

    "3M, Co.":                             "3M Co.",
    "3M Innovative Properties, Co.":       "3M Co.",
    "Minnesota Mining and Manufacturing, Co.": "3M Co.",

    "Facebook, Inc.":                      "Meta Platforms Inc.",

    "Yahoo!, Inc.":                        "Yahoo Inc.",

    "eBay, Inc.":                          "eBay Inc.",

    "Netflix, Inc.":                       "Netflix Inc.",

    "Pfizer, Inc.":                        "Pfizer Inc.",
    "Pfizer Ireland Pharmaceuticals":      "Pfizer Inc.",
    "Pfizer, Ltd.":                        "Pfizer Inc.",
    "Pfizer Pharmaceuticals, LLC":         "Pfizer Inc.",

    "Novartis Pharmaceuticals, Corp.":     "Novartis AG",
    "Novartis AG":                         "Novartis AG",
    "Novartis, Corp.":                     "Novartis AG",
    "Novartis Pharma AG":                  "Novartis AG",
    "Novartis Pharma Ag":                  "Novartis AG",
    "Novartis International Pharmaceutical, Ltd.": "Novartis AG",
    "Novartis Consumer Health, Inc.":      "Novartis AG",
    "Novartis International AG":           "Novartis AG",

    "Teva Pharmaceuticals USA, Inc.":      "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals Usa, Inc.":      "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical Industries, Ltd.": "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals Industries, Ltd.": "Teva Pharmaceutical Industries",
    "Teva Neuroscience, Inc.":             "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals U.S.A., Inc.":   "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical USA, Inc.":       "Teva Pharmaceutical Industries",

    "Mylan Pharmaceuticals, Inc.":         "Mylan Inc.",
    "Mylan, Inc.":                         "Mylan Inc.",
    "Mylan Laboratories, Inc.":            "Mylan Inc.",
    "Mylan Technologies, Inc.":            "Mylan Inc.",
    "Mylan N.V.":                          "Mylan Inc.",

    "Astrazeneca Ab":                      "AstraZeneca PLC",
    "AstraZeneca Pharmaceuticals LP":      "AstraZeneca PLC",
    "Astrazeneca Pharmaceuticals Lp":      "AstraZeneca PLC",
    "AstraZeneca AB":                      "AstraZeneca PLC",
    "Astrazeneca Uk, Ltd.":                "AstraZeneca PLC",
    "Astrazeneca Lp":                      "AstraZeneca PLC",
    "AstraZeneca UK, Ltd.":                "AstraZeneca PLC",
    "Astrazeneca AB":                      "AstraZeneca PLC",
    "AstraZeneca LP":                      "AstraZeneca PLC",
    "Astrazeneca Pharmaceuticals LP":      "AstraZeneca PLC",

    "Abbott Laboratories":                 "Abbott Laboratories",
    "Abbott Laboratories, Inc.":           "Abbott Laboratories",
    "Abbott Labs":                         "Abbott Laboratories",
    "Abbott Cardiovascular Systems, Inc.": "Abbott Laboratories",
    "Abbott Diabetes Care, Inc.":          "Abbott Laboratories",
    "Abbott Vascular, Inc.":               "Abbott Laboratories",

    "Merck & Co., Inc.":                   "Merck & Co.",
    "Merck Sharp & Dohme, Corp.":          "Merck & Co.",
    "Merck & Co, Inc.":                    "Merck & Co.",
    "Merck Sharp & Dohme B.V.":            "Merck & Co.",
    "Merck & Company, Inc.":               "Merck & Co.",

    "Johnson & Johnson":                   "Johnson & Johnson",
    "Johnson & Johnson, Inc.":             "Johnson & Johnson",
    "Johnson & Johnson Vision Care, Inc.": "Johnson & Johnson",
    "Johnson & Johnson Pharmaceutical Research & Development, LLC": "Johnson & Johnson",

    "Glaxo Group, Ltd.":                   "GlaxoSmithKline PLC",
    "Glaxo Wellcome, Inc.":                "GlaxoSmithKline PLC",
    "GlaxoSmithKline, LLC":                "GlaxoSmithKline PLC",
    "Glaxosmithkline, LLC":                "GlaxoSmithKline PLC",
    "GlaxoSmithKline":                     "GlaxoSmithKline PLC",
    "Glaxosmithkline, Inc.":               "GlaxoSmithKline PLC",
    "Glaxosmithkline PLC":                 "GlaxoSmithKline PLC",
    "GlaxoSmithKline PLC":                 "GlaxoSmithKline PLC",
    "Glaxo, Inc.":                         "GlaxoSmithKline PLC",

    "Sanofi-Aventis U.S., LLC":            "Sanofi S.A.",
    "Sanofi-Aventis":                      "Sanofi S.A.",
    "Sanofi":                              "Sanofi S.A.",
    "Sanofi-Aventis US, LLC":              "Sanofi S.A.",
    "Sanofi-Aventis U.S., Inc.":           "Sanofi S.A.",
    "Sanofi-Synthelabo, Inc.":             "Sanofi S.A.",

    "Hoffmann-La Roche, Inc.":             "Roche Holding AG",
    "Roche Palo Alto, LLC":                "Roche Holding AG",
    "Roche Molecular Systems, Inc.":       "Roche Holding AG",
    "Hoffman-La Roche, Inc.":              "Roche Holding AG",
    "Roche Diagnostics Operations, Inc.":  "Roche Holding AG",
    "Roche Diagnostics, Corp.":            "Roche Holding AG",
    "Roche Laboratories, Inc.":            "Roche Holding AG",
    "F. Hoffmann-La Roche, Ltd.":          "Roche Holding AG",
    "Hoffman-LaRoche, Inc.":               "Roche Holding AG",

    "Bristol-Myers Squibb, Co.":           "Bristol-Myers Squibb Co.",
    "Bristol-Meyers Squibb, Co.":          "Bristol-Myers Squibb Co.",
    "Bristol-Myers Squibb Pharma, Co.":    "Bristol-Myers Squibb Co.",
    "Bristol Myers Squibb, Co.":           "Bristol-Myers Squibb Co.",
    "Bristol-Myers, Co.":                  "Bristol-Myers Squibb Co.",

    "Bayer, Corp.":                        "Bayer AG",
    "Bayer Healthcare Pharmaceuticals, Inc.": "Bayer AG",
    "Bayer AG":                            "Bayer AG",
    "Bayer Schering Pharma AG":            "Bayer AG",
    "Bayer Pharma AG":                     "Bayer AG",
    "Bayer Healthcare, LLC":               "Bayer AG",
    "Bayer HealthCare Pharmaceuticals, Inc.": "Bayer AG",

    "Lupin, Ltd.":                         "Lupin Ltd.",
    "Lupin Pharmaceuticals, Inc.":         "Lupin Ltd.",
    "Lupin Atlantis Holdings S.A.":        "Lupin Ltd.",
    "Lupin, Inc.":                         "Lupin Ltd.",

    "Ranbaxy Laboratories, Ltd.":          "Ranbaxy Laboratories",
    "Ranbaxy Pharmaceuticals, Inc.":       "Ranbaxy Laboratories",
    "Ranbaxy, Inc.":                       "Ranbaxy Laboratories",
    "Ranbaxy Laboratories, Inc.":          "Ranbaxy Laboratories",

    "Watson Laboratories, Inc.":           "Actavis Inc.",
    "Watson Pharmaceuticals, Inc.":        "Actavis Inc.",
    "Watson Pharma, Inc.":                 "Actavis Inc.",
    "Actavis, Inc.":                       "Actavis Inc.",
    "Watson Pharmaceutical, Inc.":         "Actavis Inc.",

    "Sandoz, Inc.":                        "Sandoz Inc.",

    "Apotex, Inc.":                        "Apotex Inc.",
    "Apotex, Corp.":                       "Apotex Inc.",

    "Philips Electronics North America, Corp.": "Koninklijke Philips N.V.",
    "U.S. Philips, Corp.":                 "Koninklijke Philips N.V.",
    "Koninklijke Philips Electronics N.V.": "Koninklijke Philips N.V.",
    "Koninklijke Philips N.V.":            "Koninklijke Philips N.V.",
    "US Philips, Corp.":                   "Koninklijke Philips N.V.",
    "North American Philips, Corp.":       "Koninklijke Philips N.V.",

    "Xerox, Corp.":                        "Xerox Corp.",

    "NCR, Corp.":                          "NCR Corp.",

    "Texas Instruments, Inc.":             "Texas Instruments Inc.",

    "Advanced Micro Devices, Inc.":        "AMD Inc.",

    "Whirlpool, Corp.":                    "Whirlpool Corp.",

    "Emerson Electric, Co.":               "Emerson Electric Co.",

    "Honeywell International, Inc.":       "Honeywell International Inc.",

    "Stryker, Corp.":                      "Stryker Corp.",

    "Medtronic, Inc.":                     "Medtronic Inc.",

    "Boston Scientific, Corp.":            "Boston Scientific Corp.",
    "Boston Scientific Scimed, Inc.":      "Boston Scientific Corp.",

    "Genentech, Inc.":                     "Genentech Inc.",

    "Monsanto, Co.":                       "Monsanto Co.",
    "Monsanto Technology, LLC":            "Monsanto Co.",

    "Eastman Kodak, Co.":                  "Eastman Kodak Co.",

    "Garmin International, Inc.":          "Garmin Ltd.",

    "Juniper Networks, Inc.":              "Juniper Networks Inc.",

    "NEC, Corp.":                          "NEC Corp.",

    "Sharp Electronics, Corp.":            "Sharp Corp.",
    "Sharp, Corp.":                        "Sharp Corp.",

    "Hitachi, Ltd.":                       "Hitachi Ltd.",

    "Ericsson, Inc.":                      "Ericsson AB",
    "Telefonaktiebolaget LM Ericsson":     "Ericsson AB",

    "ZTE (USA), Inc.":                     "ZTE Corp.",
    "ZTE, Corp.":                          "ZTE Corp.",

    "Celgene, Corp.":                      "Celgene Corp.",

    "Allergan, Inc.":                      "Allergan Inc.",

    "Genzyme, Corp.":                      "Genzyme Corp.",

    "Illinois Tool Works, Inc.":           "Illinois Tool Works Inc.",

    "Kimberly-Clark, Corp.":               "Kimberly-Clark Corp.",

    "Ecolab, Inc.":                        "Ecolab Inc.",

    "American Express, Co.":               "American Express Co.",

    "Sears Holdings, Corp.":               "Sears Holdings Corp.",

    "Office Depot, Inc.":                  "Office Depot Inc.",

    "Staples, Inc.":                       "Staples Inc.",

    "Walgreen, Co.":                       "Walgreens Co.",

    "Rite Aid, Corp.":                     "Rite Aid Corp.",

    "Barnes & Noble, Inc.":                "Barnes & Noble Inc.",

    "GeoTag, Inc.":                        "GeoTag Inc.",
    "Geotag, Inc.":                        "GeoTag Inc.",

    "Orion Ip, LLC":                       "Orion IP, LLC",
    "Orion IP, LLC Orion IP, LLC":         "Orion IP, LLC",

    "Amazon.Com, Inc.":                    "Amazon.com Inc.",
    "Amazon.Com, Inc.,":                   "Amazon.com Inc.",
    "Amazon Web Services, LLC":            "Amazon.com Inc.",
    "Amazon.com, LLC":                     "Amazon.com Inc.",
    "Amazon Digital Services, Inc.":       "Amazon.com Inc.",
    "Amazon Services, LLC":                "Amazon.com Inc.",
    "Amazon Web Services, Inc.":           "Amazon.com Inc.",
    "Amazon Technologies, Inc.":           "Amazon.com Inc.",
    "Amazon Payments, Inc.":               "Amazon.com Inc.",
    "Amazon Services, Inc.":               "Amazon.com Inc.",
    "Amazon Com, Inc.":                    "Amazon.com Inc.",
    "Amazon, Inc.":                        "Amazon.com Inc.",
    "Amazon":                              "Amazon.com Inc.",
    "Amazon Technologies":                 "Amazon.com Inc.",
    "Amazon.Com,Inc.":                     "Amazon.com Inc.",
    "AmazonFresh, LLC":                    "Amazon.com Inc.",

    "ATT Mobility, LLC":                   "AT&T Inc.",
    "ATT, Inc.":                           "AT&T Inc.",
    "ATT, Corp.":                          "AT&T Inc.",
    "ATT Services, Inc.":                  "AT&T Inc.",
    "ATT Mobility, Corp.":                 "AT&T Inc.",
    "ATT Wireless Services, Inc.":         "AT&T Inc.",
    "ATT Intellectual Property I, L.P.":   "AT&T Inc.",
    "ATT Intellectual Property II, L.P.":  "AT&T Inc.",
    "ATT Mobility II, LLC":                "AT&T Inc.",
    "ATT Video Services, Inc.":            "AT&T Inc.",
    "Att Mobility, LLC":                   "AT&T Inc.",
    "Att":                                 "AT&T Inc.",
    "At&T, Inc.":                          "AT&T Inc.",
    "At&T Services, Inc.":                 "AT&T Inc.",

    "BestBuy.com, LLC":                    "Best Buy Co.",
    "Bestbuy.com, LLC":                    "Best Buy Co.",
    "BestBuy.Com, LLC":                    "Best Buy Co.",
    "Bestbuy.Com, LLC":                    "Best Buy Co.",
    "BestBuy Co., Inc.":                   "Best Buy Co.",
    "BestBuy, Inc.":                       "Best Buy Co.",

    "Dr. Reddy'S Laboratories, Ltd.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Laboratories, Inc.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy's Laboratories, Inc.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy's Laboratories, Ltd.":      "Dr. Reddy's Laboratories",
    "Dr. Reddys Laboratories, Ltd.":       "Dr. Reddy's Laboratories",
    "Dr. Reddys Laboratories, Inc.":       "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Pharmaceuticals, Ltd.":   "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Pharmaceuticals, Inc.":   "Dr. Reddy's Laboratories",

    "Actavis Elizabeth, LLC":              "Actavis Inc.",
    "Actavis Pharma, Inc.":                "Actavis Inc.",
    "Actavis Laboratories Fl, Inc.":       "Actavis Inc.",
    "Actavis Laboratories FL, Inc.":       "Actavis Inc.",
    "Actavis, LLC":                        "Actavis Inc.",
    "Actavis South Atlantic, LLC":         "Actavis Inc.",
    "Actavis Mid Atlantic, LLC":           "Actavis Inc.",
    "Actavis Plc":                         "Actavis Inc.",
    "Actavis plc":                         "Actavis Inc.",
    "Actavis Group":                       "Actavis Inc.",
    "Actavis Totowa, LLC":                 "Actavis Inc.",

    "Cellco Partnership":                  "Verizon Communications",
    "Cellco Partnership, Inc.":            "Verizon Communications",
    "Cellco Partnership d/b/a/ Verizon Wireless": "Verizon Communications",
    "Cellco Partnership, d/b/a Verizon Wireless": "Verizon Communications",

    "Kyocera Communications, Inc.":        "Kyocera Corp.",
    "Kyocera Wireless, Corp.":             "Kyocera Corp.",
    "Kyocera, Corp.":                      "Kyocera Corp.",
    "Kyocera International, Inc.":         "Kyocera Corp.",
    "Kyocera America, Inc.":               "Kyocera Corp.",
    "Kyocera Mita America, Inc.":          "Kyocera Corp.",
    "Kyocera Optics, Inc.":                "Kyocera Corp.",
    "Kyocera Document Solutions America, Inc.": "Kyocera Corp.",
    "Kyocera Document Solutions, Inc.":    "Kyocera Corp.",

    "Ricoh Company, Ltd.":                 "Ricoh Co. Ltd.",
    "Ricoh Americas, Corp.":               "Ricoh Co. Ltd.",
    "Ricoh, Corp.":                        "Ricoh Co. Ltd.",
    "Ricoh Electronics, Inc.":             "Ricoh Co. Ltd.",
    "Ricoh Innovations, Inc.":             "Ricoh Co. Ltd.",
    "Ricoh USA, Inc.":                     "Ricoh Co. Ltd.",
    "Ricoh Co, Ltd.":                      "Ricoh Co. Ltd.",
    "RICOH Company, Ltd.":                 "Ricoh Co. Ltd.",
    "Ricoh Company, Inc.":                 "Ricoh Co. Ltd.",

    "Matsushita Electric Industrial Co., Ltd.":     "Panasonic Corp.",
    "Matsushita Electric Corporation of America":   "Panasonic Corp.",
    "Matsushita Electric Industrial Company, Ltd.": "Panasonic Corp.",
    "Matsushita Electric Corp. Of America":         "Panasonic Corp.",
    "Matsushita Electric Industrial Co, Ltd.":      "Panasonic Corp.",
    "Matsushita Electric, Corp.":                   "Panasonic Corp.",

    "Sanyo North America, Corp.":          "Sanyo Electric Co.",
    "Sanyo Electric Co., Ltd.":            "Sanyo Electric Co.",
    "Sanyo Electric Co, Ltd.":             "Sanyo Electric Co.",
    "Sanyo Manufacturing, Corp.":          "Sanyo Electric Co.",
    "Sanyo Fisher, Co.":                   "Sanyo Electric Co.",
    "Sanyo North America":                 "Sanyo Electric Co.",
    "Sanyo Electric Company, Ltd.":        "Sanyo Electric Co.",
    "Sanyo, Corp.":                        "Sanyo Electric Co.",

    "UCB, Inc.":                           "UCB S.A.",
    "UCB Pharma GmbH":                     "UCB S.A.",
    "Ucb S.A.":                            "UCB S.A.",
    "UCB Pharma, Inc.":                    "UCB S.A.",
    "UCB Societe Anonyme":                 "UCB S.A.",
    "Ucb, Inc.":                           "UCB S.A.",
    "UCB Manufacturing, Inc.":             "UCB S.A.",
    "UCB Pharma S.A.":                     "UCB S.A.",

    "Aventis Pharmaceuticals, Inc.":       "Sanofi S.A.",
    "Aventis Pharma S.A.":                 "Sanofi S.A.",
    "Aventis, Inc.":                       "Sanofi S.A.",
    "Aventis Pharma SA":                   "Sanofi S.A.",
    "Aventis Pasteur, Inc.":               "Sanofi S.A.",
    "Aventis Pasteur, Ltd.":               "Sanofi S.A.",

    "Schering, Corp.":                     "Merck & Co.",
    "Schering-Plough, Corp.":              "Merck & Co.",
    "Schering-Plough Healthcare Products, Inc.": "Merck & Co.",
    "Schering-Plough Research Institute":  "Merck & Co.",
    "Schering Plough, Corp.":              "Merck & Co.",

    "Wyeth":                               "Pfizer Inc.",
    "Wyeth, LLC":                          "Pfizer Inc.",
    "Wyeth Pharmaceuticals, Inc.":         "Pfizer Inc.",
    "Wyeth Holdings, Corp.":               "Pfizer Inc.",
    "Wyeth Laboratories, Inc.":            "Pfizer Inc.",
    "Wyeth-Ayerst Pharmaceuticals, Inc.":  "Pfizer Inc.",

    "Lucent Technologies, Inc.":           "Nokia Corp.",
    "Alcatel-Lucent USA, Inc.":            "Nokia Corp.",
    "Alcatel-Lucent Holdings, Inc.":       "Nokia Corp.",
    "Alcatel-Lucent":                      "Nokia Corp.",
    "Alcatel-Lucent S.A.":                 "Nokia Corp.",
    "Alcatel-Lucent, Inc.":                "Nokia Corp.",
    "Alcatel Lucent USA, Inc.":            "Nokia Corp.",
    "Alcatel Lucent SA":                   "Nokia Corp.",

    "Sun Microsystems, Inc.":              "Oracle Corp.",
    "Sun microsystems, Inc.":              "Oracle Corp.",
    "Sun Microsystems of California, Inc.": "Oracle Corp.",
    "Sun Microsystems Federal, Inc.":      "Oracle Corp.",

    "Palm, Inc.":                          "Palm Inc.",
    "Palm, Inc.,":                         "Palm Inc.",
    "PalmOne, Inc.":                       "Palm Inc.",
    "palmOne, Inc.":                       "Palm Inc.",
    "Palmone, Inc.":                       "Palm Inc.",
    "Palm Computing, Inc.":                "Palm Inc.",

    "Medtronic Vascular, Inc.":            "Medtronic Inc.",
    "Medtronic USA, Inc.":                 "Medtronic Inc.",
    "Medtronic Sofamor Danek USA, Inc.":   "Medtronic Inc.",
    "Medtronic Xomed, Inc.":               "Medtronic Inc.",
    "Medtronic Minimed, Inc.":             "Medtronic Inc.",
    "Medtronic MiniMed, Inc.":             "Medtronic Inc.",
    "Medtronic Navigation, Inc.":          "Medtronic Inc.",
    "Medtronics, Inc.":                    "Medtronic Inc.",

    "Alcon Laboratories, Inc.":            "Alcon Inc.",
    "Alcon Research, Ltd.":                "Alcon Inc.",
    "Alcon Pharmaceuticals, Ltd.":         "Alcon Inc.",
    "Alcon, Inc.":                         "Alcon Inc.",
    "Alcon Surgical, Inc.":                "Alcon Inc.",
    "Alcon Manufacturing, Ltd.":           "Alcon Inc.",
    "Alcon Laboratories Holding, Corp.":   "Alcon Inc.",
    "Alcon Universal, Ltd.":               "Alcon Inc.",

    "Eli Lilly And, Co.":                  "Eli Lilly and Co.",
    "Eli Lilly and, Co.":                  "Eli Lilly and Co.",
    "Eli Lilly &, Co.":                    "Eli Lilly and Co.",
    "Eli Lilly & Co., Inc.":               "Eli Lilly and Co.",
    "Eli Lilly & Company, Inc.":           "Eli Lilly and Co.",

    "Toyota Motor, Corp.":                 "Toyota Motor Corp.",
    "Toyota Motor North America, Inc.":    "Toyota Motor Corp.",
    "Toyota Motor Sales, U.S.A., Inc.":    "Toyota Motor Corp.",
    "Toyota Motor Sales USA, Inc.":        "Toyota Motor Corp.",
    "Toyota Motor Engineering & Manufacturing North America, Inc.": "Toyota Motor Corp.",
    "Toyota Motor Manufacturing, Kentucky, Inc.": "Toyota Motor Corp.",
    "Toyota Motor, Co.":                   "Toyota Motor Corp.",
    "Toyota Motors, Corp.":                "Toyota Motor Corp.",
    "Toyota Motor Manufacturing USA, Inc.": "Toyota Motor Corp.",

    "Volkswagen Group of America, Inc.":   "Volkswagen AG",
    "Volkswagen of America, Inc.":         "Volkswagen AG",
    "Volkswagen of America":               "Volkswagen AG",
    "Volkswagen Group Of America, Inc.":   "Volkswagen AG",
    "Volkswagen Of America, Inc.":         "Volkswagen AG",

    "Kia Motors America, Inc.":            "Kia Motors Corp.",
    "Kia Motors, Corp.":                   "Kia Motors Corp.",
    "Kia Motors Manufacturing Georgia, Inc.": "Kia Motors Corp.",
    "KIA Motors America, Inc.":            "Kia Motors Corp.",
    "Kia Motors Co, Ltd.":                 "Kia Motors Corp.",
    "KIA Motors, Corp.":                   "Kia Motors Corp.",
    "Kia Motors, Co.":                     "Kia Motors Corp.",

    "ASUS Computer International":         "ASUSTeK Computer Inc.",
    "Asustek Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asus Computer International":         "ASUSTeK Computer Inc.",
    "Asus Computer International, Inc.":   "ASUSTeK Computer Inc.",
    "ASUSTeK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "ASUSTek Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asus Computer Intl":                  "ASUSTeK Computer Inc.",
    "ASUS Computer International, Inc.":   "ASUSTeK Computer Inc.",
    "ASUS Computer International, Corp.":  "ASUSTeK Computer Inc.",
    "Asus Computer International, Corp.":  "ASUSTeK Computer Inc.",
    "ASUSteK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "ASUSTEK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asustek Computer,Inc":                "ASUSTeK Computer Inc.",
    "ASUSTeK COMPUTER, Inc.":              "ASUSTeK Computer Inc.",

    "Lenovo (United States), Inc.":        "Lenovo Group Ltd.",
    "Lenovo Group, Ltd.":                  "Lenovo Group Ltd.",
    "Lenovo Holding Company, Inc.":        "Lenovo Group Ltd.",
    "Lenovo, Inc.":                        "Lenovo Group Ltd.",
    "Lenovo United States, Inc.":          "Lenovo Group Ltd.",
    "Lenovo (USA), Inc.":                  "Lenovo Group Ltd.",
    "Lenovo Group":                        "Lenovo Group Ltd.",
    "Lenovo International":                "Lenovo Group Ltd.",
    "Lenovo Group., Ltd.":                 "Lenovo Group Ltd.",

    "Sears, Roebuck and, Co.":             "Sears Holdings Corp.",
    "Sears Roebuck &, Co.":                "Sears Holdings Corp.",
    "Sears Brands, LLC":                   "Sears Holdings Corp.",
    "Sears Holdings Management, Corp.":    "Sears Holdings Corp.",
    "Sears Holding, Corp.":                "Sears Holdings Corp.",
    "Sears Roebuck And, Co.":              "Sears Holdings Corp.",

    "Time Warner Cable, Inc.":             "Time Warner Inc.",
    "Time Warner, Inc.":                   "Time Warner Inc.",
    "Time Warner Cable, LLC":              "Time Warner Inc.",
    "Time Warner Entertainment Company L P": "Time Warner Inc.",
    "AOL Time Warner, Inc.":               "Time Warner Inc.",

    "Warner-Lambert Company, LLC":         "Pfizer Inc.",
    "Warner-Lambert, Co.":                 "Pfizer Inc.",
    "Warner Lambert Company, LLC":         "Pfizer Inc.",
    "Warner Lambert, Co.":                 "Pfizer Inc.",

    "Par Pharmaceutical, Inc.":            "Par Pharmaceutical Inc.",
    "Par Pharmaceutical Companies, Inc.":  "Par Pharmaceutical Inc.",

    "Procter & Gamble, Co.":               "Procter & Gamble Co.",

    "Mitsubishi Electric, Corp.":          "Mitsubishi Electric Corp.",

    "Colgate-Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate-Palmolive Co., Inc.":         "Colgate-Palmolive Co.",
    "Colgate Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate Palmolive Co., Inc.":         "Colgate-Palmolive Co.",
    "Colgate/Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate-Palmolive Company, Inc.":     "Colgate-Palmolive Co.",

    "Smithkline Beecham, Corp.":           "GlaxoSmithKline PLC",
    "SmithKline Beecham, Corp.":           "GlaxoSmithKline PLC",
    "Smithkline Beecham Plc":              "GlaxoSmithKline PLC",
    "Smithkline Beecham, P.L.C.":          "GlaxoSmithKline PLC",
    "SmithKline Beecham (Cork), Ltd.":     "GlaxoSmithKline PLC",
    "Smithkline Beecham Consumer Healthcare, L.P.": "GlaxoSmithKline PLC",
    "Glaxosmithkline":                     "GlaxoSmithKline PLC",
    "Glaxosmithkline Plc":                 "GlaxoSmithKline PLC",
    "Glaxosmithkline Consumer Healthcare, L.P.": "GlaxoSmithKline PLC",
    "Glaxosmithkline, Corp.":              "GlaxoSmithKline PLC",
    "GLAXOSMITHKLINE p.l.c.":             "GlaxoSmithKline PLC",

    "Forest Laboratories Holdings, Ltd.":  "Forest Laboratories Inc.",
    "Forest Laboratories, Inc.":           "Forest Laboratories Inc.",
    "Forest Laboratories, LLC":            "Forest Laboratories Inc.",
    "Forest Laboratories Holding, Ltd.":   "Forest Laboratories Inc.",

    "Barr Laboratories, Inc.":             "Barr Laboratories Inc.",
    "Barr Laboratories, Inc.,":            "Barr Laboratories Inc.",
    "Barr Laboratories":                   "Barr Laboratories Inc.",

    "Hospira, Inc.":                       "Hospira Inc.",
    "Hospira Australia Pty, Ltd.":         "Hospira Inc.",
    "Hospira Worldwide, Inc.":             "Hospira Inc.",

    "Cordis, Corp.":                       "Cordis Corp.",
    "Cordis, LLC":                         "Cordis Corp.",
    "Cordis Endovascular Systems, Inc.":   "Cordis Corp.",

    "Gateway, Inc.":                       "Gateway Inc.",
    "Gateway Companies, Inc.":             "Gateway Inc.",
    "Gateway Country Stores, LLC":         "Gateway Inc.",
    "Gateway 2000, Inc.":                  "Gateway Inc.",

    "Freescale Semiconductor, Inc.":       "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Holdings I, Ltd.": "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Japan, Ltd.": "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Pte., Ltd.":  "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Taiwan, Ltd.": "Freescale Semiconductor Inc.",

    "Endo Pharmaceuticals, Inc.":          "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Solutions, Inc.": "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Holding, Inc.":  "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Holdings, Inc.": "Endo Pharmaceuticals Inc.",

    "Baxter Healthcare, Corp.":            "Baxter International Inc.",
    "Baxter International, Inc.":          "Baxter International Inc.",
    "Baxter Healthcare S.A.":              "Baxter International Inc.",
    "Baxter Diagnostics, Inc.":            "Baxter International Inc.",
    "Baxter Pharmaceutical Products, Inc.": "Baxter International Inc.",
    "Baxter Healthcare, Co.":              "Baxter International Inc.",

    "Black & Decker (U.S.), Inc.":         "Stanley Black & Decker Inc.",
    "Black & Decker, Inc.":                "Stanley Black & Decker Inc.",
    "Black & Decker, Corp.":               "Stanley Black & Decker Inc.",
    "Stanley Black & Decker, Inc.":        "Stanley Black & Decker Inc.",
    "Black & Decker (US), Inc.":           "Stanley Black & Decker Inc.",
    "The Black & Decker, Corp.":           "Stanley Black & Decker Inc.",
    "Black & Decker U.S., Inc.":           "Stanley Black & Decker Inc.",

    "Robert Bosch, LLC":                   "Robert Bosch GmbH",
    "Robert Bosch GmbH":                   "Robert Bosch GmbH",
    "Robert Bosch, Corp.":                 "Robert Bosch GmbH",
    "Robert Bosch Healthcare Systems, Inc.": "Robert Bosch GmbH",
    "Robert Bosch Tool, Corp.":            "Robert Bosch GmbH",
    "Robert Bosch North America, Corp.":   "Robert Bosch GmbH",
    "Robert Bosch Power Tool, Corp.":      "Robert Bosch GmbH",
    "Robert Bosch Gmbh":                   "Robert Bosch GmbH",
    "Bosch Security Systems, Inc.":        "Robert Bosch GmbH",

    "The Procter & Gamble, Co.":           "Procter & Gamble Co.",
    "Procter & Gamble Distributing, Co.":  "Procter & Gamble Co.",
    "Procter & Gamble Manufacturing, Co.": "Procter & Gamble Co.",
    "Procter and Gamble, Co.":             "Procter & Gamble Co.",

    "Allergan Sales, LLC":                 "Allergan Inc.",
    "Allergan USA, Inc.":                  "Allergan Inc.",
    "Allergan Sales, Inc.":                "Allergan Inc.",
    "Allergan Plc":                        "Allergan Inc.",
    "Allergan Optical, Inc.":              "Allergan Inc.",
    "Allergan Usa, Inc.":                  "Allergan Inc.",

    "Sandisk, Corp.":                      "SanDisk Corp.",
    "SanDisk, Corp.":                      "SanDisk Corp.",

    "Logitech, Inc.":                      "Logitech International S.A.",
    "Logitech International SA":           "Logitech International S.A.",
    "Logitech Europe S A":                 "Logitech International S.A.",
    "Logitech International S.A":          "Logitech International S.A.",
    "Logitech, LLC":                       "Logitech International S.A.",
    "Logitech Europe S.A.":                "Logitech International S.A.",

    "Garmin USA, Inc.":                    "Garmin Ltd.",
    "Garmin, Ltd.":                        "Garmin Ltd.",
    "Garmin, Corp.":                       "Garmin Ltd.",
    "Garmin International":                "Garmin Ltd.",
    "Garmin North America, Inc.":          "Garmin Ltd.",

    "Raytheon, Co.":                       "Raytheon Co.",
    "Raytheon Applied Signal Technology, Inc.": "Raytheon Co.",
    "Raytheon BBN Technologies, Corp.":    "Raytheon Co.",

    "Siemens, Corp.":                      "Siemens AG",
    "Siemens AG":                          "Siemens AG",
    "Siemens Medical Solutions USA, Inc.": "Siemens AG",
    "Siemens Enterprise Communications, Inc.": "Siemens AG",
    "Siemens Industry, Inc.":              "Siemens AG",
    "Siemens Medical Systems, Inc.":       "Siemens AG",
    "Siemens Energy & Automation, Inc.":   "Siemens AG",
    "Siemens Communications, Inc.":        "Siemens AG",

    "Koninklijke Philips Electronics NV":  "Koninklijke Philips N.V.",
    "U S Philips, Corp.":                  "Koninklijke Philips N.V.",
    "Koninklijke Philips Electronics N V": "Koninklijke Philips N.V.",

    "Cephalon, Inc.":                      "Teva Pharmaceutical Industries",
    "Cephalon France":                     "Teva Pharmaceutical Industries",

    "Genzyme Corp.":                       "Sanofi S.A.",
    "Genzyme Surgical Products, Corp.":    "Sanofi S.A.",

}

def get_parent_org(name: str) -> str:
    """Return canonical parent org name, or the original if not in map."""
    return PARENT_MAP.get(name, name)

names['standardized_org_name'] = names['standardized_org_name'].apply(get_parent_org)

In [75]:
location_terms = (
    r'Delaware|California|Florida|Washington|New York|Illinois|Minnesota|Texas|Utah|'
    r'Nevada|Colorado|Michigan|New Jersey|Georgia|Oregon|Arizona|Ohio|Indiana|Pennsylvania|'
    r'Massachusetts|Missouri|Wisconsin|Virginia|Maryland|Nebraska|Connecticut|Kansas|'
    r'Oklahoma|Tennessee|Iowa|Idaho|Arkansas|Kentucky|North Carolina|South Carolina|'
    r'Alabama|Mississippi|Louisiana|Hawaii|Alaska|Montana|Wyoming|North Dakota|South Dakota|'
    r'West Virginia|Rhode Island|Vermont|Maine|New Hampshire|New Mexico|'
    r'Canadian|Japanese|German|French|Korean|Australian|Taiwanese|Taiwan|'
    r'Calif|foreign|limited liability'
)

location_mask = names['standardized_org_name'].str.contains(
    rf'^\s*an?\s+(?:{location_terms}),?\s*(?:Corp\.|LLC|Co\.|L\.P\.|corporation|company|corp|limited|ltd\.?)?$',
    case=False, na=False, flags=re.IGNORECASE
)

names.loc[location_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [77]:
dba_mask = names['standardized_org_name'].str.contains(
    r'^\s*(?:doing business as|also known as)\s*$',
    case=False, na=False, flags=re.IGNORECASE
)

names.loc[dba_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [80]:
import re
import pandas as pd

# Parent company map

PARENT_MAP = {

    "Apple, Inc.":                          "Apple Inc.",
    "Apple, Corp.":                         "Apple Inc.",
    "Apple Computer, Inc.":                 "Apple Inc.",
    "Apple Computer Company, Inc.":         "Apple Inc.",
    "Apple Computers, Inc.":                "Apple Inc.",
    "Apple Computer inc":                   "Apple Inc.",
    "Apple Incorporated":                   "Apple Inc.",
    "Apple Inc,":                           "Apple Inc.",
    "APPLE INC.":                           "Apple Inc.",
    "APPLE, INC.":                          "Apple Inc.",
    "Apple Inc":                            "Apple Inc.",

    "Samsung Electronics America, Inc.":            "Samsung Electronics",
    "Samsung Electronics Co., Ltd.":                "Samsung Electronics",
    "Samsung Telecommunications America, LLC":      "Samsung Electronics",
    "Samsung Semiconductor, Inc.":                  "Samsung Electronics",
    "Samsung Electronics Co, Ltd.":                 "Samsung Electronics",
    "Samsung TeleCommunications America, LLC":       "Samsung Electronics",
    "Samsung Electronics Co., LTD.,":               "Samsung Electronics",
    "Samsung Austin Semiconductor, LLC":            "Samsung Electronics",
    "Samsung Telecommunications America LLP":       "Samsung Electronics",
    "Samsung Electronics America, Inc.,":           "Samsung Electronics",
    "Samsung Electronics Company, Ltd.":            "Samsung Electronics",
    "Samsung Electronics, Co., Ltd.":               "Samsung Electronics",
    "Samsung SDI Co., Ltd.":                        "Samsung Electronics",
    "Samsung Opto-Electronics America, Inc.":       "Samsung Electronics",
    "Samsung Electronics USA, Inc.":                "Samsung Electronics",
    "Samsung Techwin Co., Ltd.":                    "Samsung Electronics",
    "Samsung SDI Co, Ltd.":                         "Samsung Electronics",
    "Samsung Telecommunications America, Inc.":     "Samsung Electronics",
    "Samsung America, Inc.":                        "Samsung Electronics",
    "Samsung Electronics America":                  "Samsung Electronics",
    "Samsung Electronics":                          "Samsung Electronics",
    "Samsung, Inc.":                                "Samsung Electronics",
    "Samsung, Corp.":                               "Samsung Electronics",
    "Samsung Electronics, Inc.":                    "Samsung Electronics",
    "Samsung Display Co., Ltd.":                    "Samsung Electronics",
    "Samsung Mobile Display Co., Ltd.":             "Samsung Electronics",
    "Samsung Electro-Mechanics Co., Ltd.":          "Samsung Electronics",
    "Samsung TeleCommunications America, Inc.":     "Samsung Electronics",
    "Samsung Mobile":                               "Samsung Electronics",

    "Sony Electronics, Inc.":                       "Sony Corp.",
    "Sony, Corp.":                                  "Sony Corp.",
    "Sony Corporation of America":                  "Sony Corp.",
    "Sony Ericsson Mobile Communications (USA), Inc.": "Sony Corp.",
    "Sony Computer Entertainment America, Inc.":    "Sony Corp.",
    "Sony Mobile Communications (USA), Inc.":       "Sony Corp.",
    "Sony Computer Entertainment America, LLC":     "Sony Corp.",
    "Sony Computer Entertainment, Inc.":            "Sony Corp.",
    "Sony Mobile Communications AB":                "Sony Corp.",
    "Sony BMG Music Entertainment":                 "Sony Corp.",
    "Sony DADC US, Inc.":                           "Sony Corp.",
    "Sony Online Entertainment, LLC":               "Sony Corp.",
    "Sony Music Entertainment, Inc.":               "Sony Corp.",
    "Sony Pictures Entertainment, Inc.":            "Sony Corp.",
    "Sony Corp of America":                         "Sony Corp.",
    "Sony Corporation Of America":                  "Sony Corp.",
    "Sony Interactive Entertainment, Inc.":         "Sony Corp.",

    "HTC America, Inc.":                    "HTC Corp.",
    "HTC, Corp.":                           "HTC Corp.",
    "HTC (BVI), Corp.":                     "HTC Corp.",
    "HTC BVI, Corp.":                       "HTC Corp.",
    "HTC America, Inc.,":                   "HTC Corp.",
    "HTC (B.V.I.), Corp.":                  "HTC Corp.",
    "HTC America Holding, Inc.":            "HTC Corp.",
    "HTC USA, Inc.":                        "HTC Corp.",
    "HTC America Innovation, Inc.":         "HTC Corp.",
    "HTC, Inc.":                            "HTC Corp.",
    "HTC America":                          "HTC Corp.",
    "HTC America Holdings, Inc.":           "HTC Corp.",

    "LG Electronics, Inc.":                 "LG Electronics",
    "LG Electronics USA, Inc.":             "LG Electronics",
    "LG Electronics U.S.A., Inc.":          "LG Electronics",
    "LG Electronics Mobilecomm U.S.A., Inc.": "LG Electronics",
    "LG Electronics Mobilecomm USA, Inc.":  "LG Electronics",
    "LG Electronics MobileComm U.S.A., Inc.": "LG Electronics",
    "LG Electronics MobileComm USA, Inc.":  "LG Electronics",
    "LG Display Co., Ltd.":                 "LG Electronics",
    "LG Display America, Inc.":             "LG Electronics",
    "LG Electronics U.S.A., Inc.,":         "LG Electronics",
    "LG Electronics":                       "LG Electronics",
    "LG Electronics, U.S.A., Inc.":         "LG Electronics",

    "Microsoft, Corp.":                     "Microsoft Corp.",
    "Microsoft, Corp":                      "Microsoft Corp.",

    "Google, Inc.":                         "Google Inc.",

    "Amazon.com, Inc.":                     "Amazon.com Inc.",

    "Dell, Inc.":                           "Dell Inc.",

    "Intel, Corp.":                         "Intel Corp.",

    "Cisco Systems, Inc.":                  "Cisco Systems Inc.",

    "Qualcomm, Inc.":                       "Qualcomm Inc.",
    "Qualcomm Technologies, Inc.":          "Qualcomm Inc.",
    "Qualcomm Atheros, Inc.":               "Qualcomm Inc.",
    "QUALCOMM, Inc.":                       "Qualcomm Inc.",
    "Qualcomm-Atheros, Inc.":               "Qualcomm Inc.",
    "Qualcomm Innovation Center, Inc.":     "Qualcomm Inc.",

    "Broadcom, Corp.":                      "Broadcom Corp.",
    "BROADCOM, Corp.":                      "Broadcom Corp.",

    "Motorola, Inc.":                       "Motorola Inc.",
    "Motorola Mobility, Inc.":              "Motorola Inc.",
    "Motorola Mobility, LLC":               "Motorola Inc.",
    "Motorola Solutions, Inc.":             "Motorola Inc.",
    "Motorola Mobility Holdings, Inc.":     "Motorola Inc.",
    "Motorola, Corp.":                      "Motorola Inc.",
    "Motorola Mobility Holdings, LLC":      "Motorola Inc.",
    "Motorola Mobility International, Ltd.": "Motorola Inc.",

    "Nokia, Inc.":                          "Nokia Corp.",
    "Nokia, Corp.":                         "Nokia Corp.",
    "Nokia Mobile Phones, Inc.":            "Nokia Corp.",
    "Nokia Holding, Inc.":                  "Nokia Corp.",
    "Nokia Siemens Networks US, LLC":       "Nokia Corp.",
    "Nokia Mobile Phones Americas, Inc.":   "Nokia Corp.",
    "Nokia USA, Inc.":                      "Nokia Corp.",
    "Nokia Mobile Phones, Ltd.":            "Nokia Corp.",
    "Nokia Networks, Inc.":                 "Nokia Corp.",
    "Nokia Products, Corp.":                "Nokia Corp.",
    "Nokia Mobile Phones":                  "Nokia Corp.",

    "BlackBerry, Corp.":                    "BlackBerry Ltd.",
    "BlackBerry, Ltd.":                     "BlackBerry Ltd.",
    "Blackberry, Ltd.":                     "BlackBerry Ltd.",
    "Blackberry, Corp.":                    "BlackBerry Ltd.",
    "Research In Motion, Corp.":            "BlackBerry Ltd.",
    "Research In Motion, Ltd.":             "BlackBerry Ltd.",
    "Research in Motion, Ltd.":             "BlackBerry Ltd.",
    "Research in Motion, Corp.":            "BlackBerry Ltd.",
    "Research In Motion, Inc.":             "BlackBerry Ltd.",

    "Huawei Device USA, Inc.":              "Huawei Technologies",
    "Huawei Technologies USA, Inc.":        "Huawei Technologies",
    "Huawei Technologies Co., Ltd.":        "Huawei Technologies",
    "Huawei Devices USA, Inc.":             "Huawei Technologies",
    "Huawei Technologies Co, Ltd.":         "Huawei Technologies",
    "Huawei Device Co., Ltd.":              "Huawei Technologies",
    "Huawei America, Inc.":                 "Huawei Technologies",
    "Huawei Technologies, Co., Ltd.":       "Huawei Technologies",
    "Huawei Technologies USA":              "Huawei Technologies",

    "Toshiba, Corp.":                       "Toshiba Corp.",
    "Toshiba America Information Systems, Inc.": "Toshiba Corp.",
    "Toshiba America, Inc.":                "Toshiba Corp.",
    "Toshiba America Electronic Components, Inc.": "Toshiba Corp.",
    "Toshiba America Consumer Products, LLC": "Toshiba Corp.",
    "Toshiba America Medical Systems, Inc.": "Toshiba Corp.",
    "Toshiba America Business Solutions, Inc.": "Toshiba Corp.",
    "Toshiba International, Corp.":         "Toshiba Corp.",

    "Panasonic Corporation of North America": "Panasonic Corp.",
    "Panasonic, Corp.":                     "Panasonic Corp.",
    "Panasonic Corp. of North America":     "Panasonic Corp.",
    "Panasonic Corporation Of North America": "Panasonic Corp.",
    "Panasonic Consumer Electronics, Co.":  "Panasonic Corp.",
    "Panasonic Corporation of North America, Inc.": "Panasonic Corp.",
    "Panasonic Corporation of America":     "Panasonic Corp.",

    "AT&T Mobility, LLC":                   "AT&T Inc.",
    "AT&T, Inc.":                           "AT&T Inc.",
    "AT&T, Corp.":                          "AT&T Inc.",
    "AT&T Services, Inc.":                  "AT&T Inc.",
    "AT&T Operations, Inc.":                "AT&T Inc.",
    "AT&T Wireless Services, Inc.":         "AT&T Inc.",
    "AT&T Mobility II, LLC":                "AT&T Inc.",
    "AT&T Internet Services":               "AT&T Inc.",
    "At&T, Corp.":                          "AT&T Inc.",
    "At&T Mobility, LLC":                   "AT&T Inc.",
    "AT&T Mobility, Corp.":                 "AT&T Inc.",

    "Verizon Communications, Inc.":         "Verizon Communications",
    "Cellco Partnership d/b/a Verizon Wireless": "Verizon Communications",
    "Verizon Services, Corp.":              "Verizon Communications",
    "Verizon Wireless":                     "Verizon Communications",
    "Verizon Business Network Services, Inc.": "Verizon Communications",
    "Verizon Online, LLC":                  "Verizon Communications",
    "Verizon Data Services, LLC":           "Verizon Communications",
    "Verizon South, Inc.":                  "Verizon Communications",
    "Verizon Wireless, Inc.":               "Verizon Communications",
    "Verizon Virginia, Inc.":               "Verizon Communications",
    "Verizon California, Inc.":             "Verizon Communications",

    "Sprint Spectrum, L.P.":                "Sprint Corp.",
    "Sprint Nextel, Corp.":                 "Sprint Corp.",
    "Sprint Solutions, Inc.":               "Sprint Corp.",
    "Sprint, Corp.":                        "Sprint Corp.",
    "Sprint Communications Company, L.P.":  "Sprint Corp.",
    "Sprint Communications, Inc.":          "Sprint Corp.",
    "Sprint Spectrum LP":                   "Sprint Corp.",

    "T-Mobile USA, Inc.":                   "T-Mobile USA Inc.",

    "Comcast, Corp.":                       "Comcast Corp.",
    "Comcast Cable Communications, LLC":    "Comcast Corp.",
    "Comcast Interactive Media, LLC":       "Comcast Corp.",

    "Hewlett-Packard, Co.":                 "HP Inc.",
    "Hewlett Packard, Co.":                 "HP Inc.",
    "Hewlett-Packard Development Company, L.P.": "HP Inc.",
    "Hewlett-Packard, Corp.":               "HP Inc.",
    "Hewlett Packard Enterprise, Co.":      "HP Inc.",
    "Hewlett Packard, Corp.":               "HP Inc.",

    "International Business Machines, Corp.": "IBM Corp.",

    "Oracle, Corp.":                        "Oracle Corp.",

    "Adobe Systems, Inc.":                  "Adobe Inc.",

    "Symantec, Corp.":                      "Symantec Corp.",

    "Canon, Inc.":                          "Canon Inc.",
    "Canon U.S.A., Inc.":                   "Canon Inc.",
    "Canon USA, Inc.":                      "Canon Inc.",
    "Canon Solutions America, Inc.":        "Canon Inc.",
    "Canon Computer Systems, Inc.":         "Canon Inc.",

    "Fujitsu, Ltd.":                        "Fujitsu Ltd.",
    "Fujitsu America, Inc.":                "Fujitsu Ltd.",
    "Fujitsu Computer Products of America, Inc.": "Fujitsu Ltd.",
    "Fujitsu Network Communications, Inc.": "Fujitsu Ltd.",
    "Fujitsu Computer Systems, Corp.":      "Fujitsu Ltd.",
    "Fujitsu Microelectronics, Inc.":       "Fujitsu Ltd.",
    "Fujitsu Semiconductor, Ltd.":          "Fujitsu Ltd.",
    "Fujitsu Semiconductor America, Inc.":  "Fujitsu Ltd.",

    "Epson America, Inc.":                  "Seiko Epson Corp.",
    "Seiko Epson, Corp.":                   "Seiko Epson Corp.",

    "Nintendo of America, Inc.":            "Nintendo Co. Ltd.",
    "Nintendo Co., Ltd.":                   "Nintendo Co. Ltd.",
    "Nintendo Co, Ltd.":                    "Nintendo Co. Ltd.",
    "Nintendo Company, Ltd.":               "Nintendo Co. Ltd.",
    "Nintendo Of America, Inc.":            "Nintendo Co. Ltd.",
    "Nintendo of America":                  "Nintendo Co. Ltd.",
    "Nintendo of North America, Inc.":      "Nintendo Co. Ltd.",

    "Acer America, Corp.":                  "Acer Inc.",
    "Acer, Inc.":                           "Acer Inc.",

    "General Electric, Co.":               "General Electric Co.",
    "General Electric Capital, Corp.":     "General Electric Co.",
    "General Electric, Corp.":             "General Electric Co.",
    "The General Electric, Co.":           "General Electric Co.",

    "General Motors, Corp.":               "General Motors Co.",
    "General Motors, LLC":                 "General Motors Co.",
    "General Motors, Co.":                 "General Motors Co.",
    "General Motors Acceptance, Corp.":    "General Motors Co.",
    "General Motors of Canada, Ltd.":      "General Motors Co.",

    "Ford Motor, Co.":                     "Ford Motor Co.",

    "BMW of North America, LLC":           "BMW AG",
    "Bayerische Motoren Werke AG":         "BMW AG",

    "Mercedes-Benz USA, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz U.S. International, Inc.": "Mercedes-Benz Group",
    "Mercedes Benz USA, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz Usa, LLC":              "Mercedes-Benz Group",
    "Mercedes-Benz of North America, Inc.": "Mercedes-Benz Group",

    "Nissan North America, Inc.":          "Nissan Motor Co.",
    "Nissan Motor Co., Ltd.":              "Nissan Motor Co.",
    "Nissan Motor Co, Ltd.":               "Nissan Motor Co.",
    "Nissan Motor Company, Ltd.":          "Nissan Motor Co.",
    "Nissan Motor, Co.":                   "Nissan Motor Co.",

    "Hyundai Motor America":               "Hyundai Motor Co.",
    "Hyundai Motor, Co.":                  "Hyundai Motor Co.",
    "Hyundai Motor America, Inc.":         "Hyundai Motor Co.",
    "Hyundai Motor Manufacturing Alabama, LLC": "Hyundai Motor Co.",

    "Wal-Mart Stores, Inc.":               "Walmart Inc.",
    "Wal-Mart Stores Texas, LLC":          "Walmart Inc.",
    "Wal-Mart, Inc.":                      "Walmart Inc.",
    "Wal-Mart Stores East, LP":            "Walmart Inc.",
    "Wal-Mart Stores":                     "Walmart Inc.",

    "Target, Corp.":                       "Target Corp.",

    "Best Buy Co., Inc.":                  "Best Buy Co.",
    "Best Buy Stores, L.P.":               "Best Buy Co.",
    "Best Buy Co, Inc.":                   "Best Buy Co.",
    "Best Buy Company, Inc.":              "Best Buy Co.",
    "Best Buy, Co.":                       "Best Buy Co.",

    "Costco Wholesale, Corp.":             "Costco Wholesale Corp.",

    "Nike, Inc.":                          "Nike Inc.",

    "3M, Co.":                             "3M Co.",
    "3M Innovative Properties, Co.":       "3M Co.",
    "Minnesota Mining and Manufacturing, Co.": "3M Co.",

    "Facebook, Inc.":                      "Meta Platforms Inc.",

    "Yahoo!, Inc.":                        "Yahoo Inc.",

    "eBay, Inc.":                          "eBay Inc.",

    "Netflix, Inc.":                       "Netflix Inc.",

    "Pfizer, Inc.":                        "Pfizer Inc.",
    "Pfizer Ireland Pharmaceuticals":      "Pfizer Inc.",
    "Pfizer, Ltd.":                        "Pfizer Inc.",
    "Pfizer Pharmaceuticals, LLC":         "Pfizer Inc.",

    "Novartis Pharmaceuticals, Corp.":     "Novartis AG",
    "Novartis AG":                         "Novartis AG",
    "Novartis, Corp.":                     "Novartis AG",
    "Novartis Pharma AG":                  "Novartis AG",
    "Novartis Pharma Ag":                  "Novartis AG",
    "Novartis International Pharmaceutical, Ltd.": "Novartis AG",
    "Novartis Consumer Health, Inc.":      "Novartis AG",
    "Novartis International AG":           "Novartis AG",

    "Teva Pharmaceuticals USA, Inc.":      "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals Usa, Inc.":      "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical Industries, Ltd.": "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals Industries, Ltd.": "Teva Pharmaceutical Industries",
    "Teva Neuroscience, Inc.":             "Teva Pharmaceutical Industries",
    "Teva Pharmaceuticals U.S.A., Inc.":   "Teva Pharmaceutical Industries",
    "Teva Pharmaceutical USA, Inc.":       "Teva Pharmaceutical Industries",

    "Mylan Pharmaceuticals, Inc.":         "Mylan Inc.",
    "Mylan, Inc.":                         "Mylan Inc.",
    "Mylan Laboratories, Inc.":            "Mylan Inc.",
    "Mylan Technologies, Inc.":            "Mylan Inc.",
    "Mylan N.V.":                          "Mylan Inc.",

    "Astrazeneca Ab":                      "AstraZeneca PLC",
    "AstraZeneca Pharmaceuticals LP":      "AstraZeneca PLC",
    "Astrazeneca Pharmaceuticals Lp":      "AstraZeneca PLC",
    "AstraZeneca AB":                      "AstraZeneca PLC",
    "Astrazeneca Uk, Ltd.":                "AstraZeneca PLC",
    "Astrazeneca Lp":                      "AstraZeneca PLC",
    "AstraZeneca UK, Ltd.":                "AstraZeneca PLC",
    "Astrazeneca AB":                      "AstraZeneca PLC",
    "AstraZeneca LP":                      "AstraZeneca PLC",
    "Astrazeneca Pharmaceuticals LP":      "AstraZeneca PLC",

    "Abbott Laboratories":                 "Abbott Laboratories",
    "Abbott Laboratories, Inc.":           "Abbott Laboratories",
    "Abbott Labs":                         "Abbott Laboratories",
    "Abbott Cardiovascular Systems, Inc.": "Abbott Laboratories",
    "Abbott Diabetes Care, Inc.":          "Abbott Laboratories",
    "Abbott Vascular, Inc.":               "Abbott Laboratories",

    "Merck & Co., Inc.":                   "Merck & Co.",
    "Merck Sharp & Dohme, Corp.":          "Merck & Co.",
    "Merck & Co, Inc.":                    "Merck & Co.",
    "Merck Sharp & Dohme B.V.":            "Merck & Co.",
    "Merck & Company, Inc.":               "Merck & Co.",

    "Johnson & Johnson":                   "Johnson & Johnson",
    "Johnson & Johnson, Inc.":             "Johnson & Johnson",
    "Johnson & Johnson Vision Care, Inc.": "Johnson & Johnson",
    "Johnson & Johnson Pharmaceutical Research & Development, LLC": "Johnson & Johnson",

    "Glaxo Group, Ltd.":                   "GlaxoSmithKline PLC",
    "Glaxo Wellcome, Inc.":                "GlaxoSmithKline PLC",
    "GlaxoSmithKline, LLC":                "GlaxoSmithKline PLC",
    "Glaxosmithkline, LLC":                "GlaxoSmithKline PLC",
    "GlaxoSmithKline":                     "GlaxoSmithKline PLC",
    "Glaxosmithkline, Inc.":               "GlaxoSmithKline PLC",
    "Glaxosmithkline PLC":                 "GlaxoSmithKline PLC",
    "GlaxoSmithKline PLC":                 "GlaxoSmithKline PLC",
    "Glaxo, Inc.":                         "GlaxoSmithKline PLC",

    "Sanofi-Aventis U.S., LLC":            "Sanofi S.A.",
    "Sanofi-Aventis":                      "Sanofi S.A.",
    "Sanofi":                              "Sanofi S.A.",
    "Sanofi-Aventis US, LLC":              "Sanofi S.A.",
    "Sanofi-Aventis U.S., Inc.":           "Sanofi S.A.",
    "Sanofi-Synthelabo, Inc.":             "Sanofi S.A.",

    "Hoffmann-La Roche, Inc.":             "Roche Holding AG",
    "Roche Palo Alto, LLC":                "Roche Holding AG",
    "Roche Molecular Systems, Inc.":       "Roche Holding AG",
    "Hoffman-La Roche, Inc.":              "Roche Holding AG",
    "Roche Diagnostics Operations, Inc.":  "Roche Holding AG",
    "Roche Diagnostics, Corp.":            "Roche Holding AG",
    "Roche Laboratories, Inc.":            "Roche Holding AG",
    "F. Hoffmann-La Roche, Ltd.":          "Roche Holding AG",
    "Hoffman-LaRoche, Inc.":               "Roche Holding AG",

    "Bristol-Myers Squibb, Co.":           "Bristol-Myers Squibb Co.",
    "Bristol-Meyers Squibb, Co.":          "Bristol-Myers Squibb Co.",
    "Bristol-Myers Squibb Pharma, Co.":    "Bristol-Myers Squibb Co.",
    "Bristol Myers Squibb, Co.":           "Bristol-Myers Squibb Co.",
    "Bristol-Myers, Co.":                  "Bristol-Myers Squibb Co.",

    "Bayer, Corp.":                        "Bayer AG",
    "Bayer Healthcare Pharmaceuticals, Inc.": "Bayer AG",
    "Bayer AG":                            "Bayer AG",
    "Bayer Schering Pharma AG":            "Bayer AG",
    "Bayer Pharma AG":                     "Bayer AG",
    "Bayer Healthcare, LLC":               "Bayer AG",
    "Bayer HealthCare Pharmaceuticals, Inc.": "Bayer AG",

    "Lupin, Ltd.":                         "Lupin Ltd.",
    "Lupin Pharmaceuticals, Inc.":         "Lupin Ltd.",
    "Lupin Atlantis Holdings S.A.":        "Lupin Ltd.",
    "Lupin, Inc.":                         "Lupin Ltd.",

    "Ranbaxy Laboratories, Ltd.":          "Ranbaxy Laboratories",
    "Ranbaxy Pharmaceuticals, Inc.":       "Ranbaxy Laboratories",
    "Ranbaxy, Inc.":                       "Ranbaxy Laboratories",
    "Ranbaxy Laboratories, Inc.":          "Ranbaxy Laboratories",

    "Watson Laboratories, Inc.":           "Actavis Inc.",
    "Watson Pharmaceuticals, Inc.":        "Actavis Inc.",
    "Watson Pharma, Inc.":                 "Actavis Inc.",
    "Actavis, Inc.":                       "Actavis Inc.",
    "Watson Pharmaceutical, Inc.":         "Actavis Inc.",

    "Sandoz, Inc.":                        "Sandoz Inc.",

    "Apotex, Inc.":                        "Apotex Inc.",
    "Apotex, Corp.":                       "Apotex Inc.",

    "Philips Electronics North America, Corp.": "Koninklijke Philips N.V.",
    "U.S. Philips, Corp.":                 "Koninklijke Philips N.V.",
    "Koninklijke Philips Electronics N.V.": "Koninklijke Philips N.V.",
    "Koninklijke Philips N.V.":            "Koninklijke Philips N.V.",
    "US Philips, Corp.":                   "Koninklijke Philips N.V.",
    "North American Philips, Corp.":       "Koninklijke Philips N.V.",

    "Xerox, Corp.":                        "Xerox Corp.",

    "NCR, Corp.":                          "NCR Corp.",

    "Texas Instruments, Inc.":             "Texas Instruments Inc.",

    "Advanced Micro Devices, Inc.":        "AMD Inc.",

    "Whirlpool, Corp.":                    "Whirlpool Corp.",

    "Emerson Electric, Co.":               "Emerson Electric Co.",

    "Honeywell International, Inc.":       "Honeywell International Inc.",

    "Stryker, Corp.":                      "Stryker Corp.",

    "Medtronic, Inc.":                     "Medtronic Inc.",

    "Boston Scientific, Corp.":            "Boston Scientific Corp.",
    "Boston Scientific Scimed, Inc.":      "Boston Scientific Corp.",

    "Genentech, Inc.":                     "Genentech Inc.",

    "Monsanto, Co.":                       "Monsanto Co.",
    "Monsanto Technology, LLC":            "Monsanto Co.",

    "Eastman Kodak, Co.":                  "Eastman Kodak Co.",

    "Garmin International, Inc.":          "Garmin Ltd.",

    "Juniper Networks, Inc.":              "Juniper Networks Inc.",

    "NEC, Corp.":                          "NEC Corp.",

    "Sharp Electronics, Corp.":            "Sharp Corp.",
    "Sharp, Corp.":                        "Sharp Corp.",

    "Hitachi, Ltd.":                       "Hitachi Ltd.",

    "Ericsson, Inc.":                      "Ericsson AB",
    "Telefonaktiebolaget LM Ericsson":     "Ericsson AB",

    "ZTE (USA), Inc.":                     "ZTE Corp.",
    "ZTE, Corp.":                          "ZTE Corp.",

    "Celgene, Corp.":                      "Celgene Corp.",

    "Allergan, Inc.":                      "Allergan Inc.",

    "Genzyme, Corp.":                      "Genzyme Corp.",

    "Illinois Tool Works, Inc.":           "Illinois Tool Works Inc.",

    "Kimberly-Clark, Corp.":               "Kimberly-Clark Corp.",

    "Ecolab, Inc.":                        "Ecolab Inc.",

    "American Express, Co.":               "American Express Co.",

    "Sears Holdings, Corp.":               "Sears Holdings Corp.",

    "Office Depot, Inc.":                  "Office Depot Inc.",

    "Staples, Inc.":                       "Staples Inc.",

    "Walgreen, Co.":                       "Walgreens Co.",

    "Rite Aid, Corp.":                     "Rite Aid Corp.",

    "Barnes & Noble, Inc.":                "Barnes & Noble Inc.",

    "GeoTag, Inc.":                        "GeoTag Inc.",
    "Geotag, Inc.":                        "GeoTag Inc.",

    "Orion Ip, LLC":                       "Orion IP, LLC",
    "Orion IP, LLC Orion IP, LLC":         "Orion IP, LLC",

    "Amazon.Com, Inc.":                    "Amazon.com Inc.",
    "Amazon.Com, Inc.,":                   "Amazon.com Inc.",
    "Amazon Web Services, LLC":            "Amazon.com Inc.",
    "Amazon.com, LLC":                     "Amazon.com Inc.",
    "Amazon Digital Services, Inc.":       "Amazon.com Inc.",
    "Amazon Services, LLC":                "Amazon.com Inc.",
    "Amazon Web Services, Inc.":           "Amazon.com Inc.",
    "Amazon Technologies, Inc.":           "Amazon.com Inc.",
    "Amazon Payments, Inc.":               "Amazon.com Inc.",
    "Amazon Services, Inc.":               "Amazon.com Inc.",
    "Amazon Com, Inc.":                    "Amazon.com Inc.",
    "Amazon, Inc.":                        "Amazon.com Inc.",
    "Amazon":                              "Amazon.com Inc.",
    "Amazon Technologies":                 "Amazon.com Inc.",
    "Amazon.Com,Inc.":                     "Amazon.com Inc.",
    "AmazonFresh, LLC":                    "Amazon.com Inc.",

    "ATT Mobility, LLC":                   "AT&T Inc.",
    "ATT, Inc.":                           "AT&T Inc.",
    "ATT, Corp.":                          "AT&T Inc.",
    "ATT Services, Inc.":                  "AT&T Inc.",
    "ATT Mobility, Corp.":                 "AT&T Inc.",
    "ATT Wireless Services, Inc.":         "AT&T Inc.",
    "ATT Intellectual Property I, L.P.":   "AT&T Inc.",
    "ATT Intellectual Property II, L.P.":  "AT&T Inc.",
    "ATT Mobility II, LLC":                "AT&T Inc.",
    "ATT Video Services, Inc.":            "AT&T Inc.",
    "Att Mobility, LLC":                   "AT&T Inc.",
    "Att":                                 "AT&T Inc.",
    "At&T, Inc.":                          "AT&T Inc.",
    "At&T Services, Inc.":                 "AT&T Inc.",

    "BestBuy.com, LLC":                    "Best Buy Co.",
    "Bestbuy.com, LLC":                    "Best Buy Co.",
    "BestBuy.Com, LLC":                    "Best Buy Co.",
    "Bestbuy.Com, LLC":                    "Best Buy Co.",
    "BestBuy Co., Inc.":                   "Best Buy Co.",
    "BestBuy, Inc.":                       "Best Buy Co.",

    "Dr. Reddy'S Laboratories, Ltd.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Laboratories, Inc.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy's Laboratories, Inc.":      "Dr. Reddy's Laboratories",
    "Dr. Reddy's Laboratories, Ltd.":      "Dr. Reddy's Laboratories",
    "Dr. Reddys Laboratories, Ltd.":       "Dr. Reddy's Laboratories",
    "Dr. Reddys Laboratories, Inc.":       "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Pharmaceuticals, Ltd.":   "Dr. Reddy's Laboratories",
    "Dr. Reddy'S Pharmaceuticals, Inc.":   "Dr. Reddy's Laboratories",

    "Actavis Elizabeth, LLC":              "Actavis Inc.",
    "Actavis Pharma, Inc.":                "Actavis Inc.",
    "Actavis Laboratories Fl, Inc.":       "Actavis Inc.",
    "Actavis Laboratories FL, Inc.":       "Actavis Inc.",
    "Actavis, LLC":                        "Actavis Inc.",
    "Actavis South Atlantic, LLC":         "Actavis Inc.",
    "Actavis Mid Atlantic, LLC":           "Actavis Inc.",
    "Actavis Plc":                         "Actavis Inc.",
    "Actavis plc":                         "Actavis Inc.",
    "Actavis Group":                       "Actavis Inc.",
    "Actavis Totowa, LLC":                 "Actavis Inc.",

    "Cellco Partnership":                  "Verizon Communications",
    "Cellco Partnership, Inc.":            "Verizon Communications",
    "Cellco Partnership d/b/a/ Verizon Wireless": "Verizon Communications",
    "Cellco Partnership, d/b/a Verizon Wireless": "Verizon Communications",

    "Kyocera Communications, Inc.":        "Kyocera Corp.",
    "Kyocera Wireless, Corp.":             "Kyocera Corp.",
    "Kyocera, Corp.":                      "Kyocera Corp.",
    "Kyocera International, Inc.":         "Kyocera Corp.",
    "Kyocera America, Inc.":               "Kyocera Corp.",
    "Kyocera Mita America, Inc.":          "Kyocera Corp.",
    "Kyocera Optics, Inc.":                "Kyocera Corp.",
    "Kyocera Document Solutions America, Inc.": "Kyocera Corp.",
    "Kyocera Document Solutions, Inc.":    "Kyocera Corp.",

    "Ricoh Company, Ltd.":                 "Ricoh Co. Ltd.",
    "Ricoh Americas, Corp.":               "Ricoh Co. Ltd.",
    "Ricoh, Corp.":                        "Ricoh Co. Ltd.",
    "Ricoh Electronics, Inc.":             "Ricoh Co. Ltd.",
    "Ricoh Innovations, Inc.":             "Ricoh Co. Ltd.",
    "Ricoh USA, Inc.":                     "Ricoh Co. Ltd.",
    "Ricoh Co, Ltd.":                      "Ricoh Co. Ltd.",
    "RICOH Company, Ltd.":                 "Ricoh Co. Ltd.",
    "Ricoh Company, Inc.":                 "Ricoh Co. Ltd.",

    "Matsushita Electric Industrial Co., Ltd.":     "Panasonic Corp.",
    "Matsushita Electric Corporation of America":   "Panasonic Corp.",
    "Matsushita Electric Industrial Company, Ltd.": "Panasonic Corp.",
    "Matsushita Electric Corp. Of America":         "Panasonic Corp.",
    "Matsushita Electric Industrial Co, Ltd.":      "Panasonic Corp.",
    "Matsushita Electric, Corp.":                   "Panasonic Corp.",

    "Sanyo North America, Corp.":          "Sanyo Electric Co.",
    "Sanyo Electric Co., Ltd.":            "Sanyo Electric Co.",
    "Sanyo Electric Co, Ltd.":             "Sanyo Electric Co.",
    "Sanyo Manufacturing, Corp.":          "Sanyo Electric Co.",
    "Sanyo Fisher, Co.":                   "Sanyo Electric Co.",
    "Sanyo North America":                 "Sanyo Electric Co.",
    "Sanyo Electric Company, Ltd.":        "Sanyo Electric Co.",
    "Sanyo, Corp.":                        "Sanyo Electric Co.",

    "UCB, Inc.":                           "UCB S.A.",
    "UCB Pharma GmbH":                     "UCB S.A.",
    "Ucb S.A.":                            "UCB S.A.",
    "UCB Pharma, Inc.":                    "UCB S.A.",
    "UCB Societe Anonyme":                 "UCB S.A.",
    "Ucb, Inc.":                           "UCB S.A.",
    "UCB Manufacturing, Inc.":             "UCB S.A.",
    "UCB Pharma S.A.":                     "UCB S.A.",

    "Aventis Pharmaceuticals, Inc.":       "Sanofi S.A.",
    "Aventis Pharma S.A.":                 "Sanofi S.A.",
    "Aventis, Inc.":                       "Sanofi S.A.",
    "Aventis Pharma SA":                   "Sanofi S.A.",
    "Aventis Pasteur, Inc.":               "Sanofi S.A.",
    "Aventis Pasteur, Ltd.":               "Sanofi S.A.",

    "Schering, Corp.":                     "Merck & Co.",
    "Schering-Plough, Corp.":              "Merck & Co.",
    "Schering-Plough Healthcare Products, Inc.": "Merck & Co.",
    "Schering-Plough Research Institute":  "Merck & Co.",
    "Schering Plough, Corp.":              "Merck & Co.",

    "Wyeth":                               "Pfizer Inc.",
    "Wyeth, LLC":                          "Pfizer Inc.",
    "Wyeth Pharmaceuticals, Inc.":         "Pfizer Inc.",
    "Wyeth Holdings, Corp.":               "Pfizer Inc.",
    "Wyeth Laboratories, Inc.":            "Pfizer Inc.",
    "Wyeth-Ayerst Pharmaceuticals, Inc.":  "Pfizer Inc.",

    "Lucent Technologies, Inc.":           "Nokia Corp.",
    "Alcatel-Lucent USA, Inc.":            "Nokia Corp.",
    "Alcatel-Lucent Holdings, Inc.":       "Nokia Corp.",
    "Alcatel-Lucent":                      "Nokia Corp.",
    "Alcatel-Lucent S.A.":                 "Nokia Corp.",
    "Alcatel-Lucent, Inc.":                "Nokia Corp.",
    "Alcatel Lucent USA, Inc.":            "Nokia Corp.",
    "Alcatel Lucent SA":                   "Nokia Corp.",

    "Sun Microsystems, Inc.":              "Oracle Corp.",
    "Sun microsystems, Inc.":              "Oracle Corp.",
    "Sun Microsystems of California, Inc.": "Oracle Corp.",
    "Sun Microsystems Federal, Inc.":      "Oracle Corp.",

    "Palm, Inc.":                          "Palm Inc.",
    "Palm, Inc.,":                         "Palm Inc.",
    "PalmOne, Inc.":                       "Palm Inc.",
    "palmOne, Inc.":                       "Palm Inc.",
    "Palmone, Inc.":                       "Palm Inc.",
    "Palm Computing, Inc.":                "Palm Inc.",

    "Medtronic Vascular, Inc.":            "Medtronic Inc.",
    "Medtronic USA, Inc.":                 "Medtronic Inc.",
    "Medtronic Sofamor Danek USA, Inc.":   "Medtronic Inc.",
    "Medtronic Xomed, Inc.":               "Medtronic Inc.",
    "Medtronic Minimed, Inc.":             "Medtronic Inc.",
    "Medtronic MiniMed, Inc.":             "Medtronic Inc.",
    "Medtronic Navigation, Inc.":          "Medtronic Inc.",
    "Medtronics, Inc.":                    "Medtronic Inc.",

    "Alcon Laboratories, Inc.":            "Alcon Inc.",
    "Alcon Research, Ltd.":                "Alcon Inc.",
    "Alcon Pharmaceuticals, Ltd.":         "Alcon Inc.",
    "Alcon, Inc.":                         "Alcon Inc.",
    "Alcon Surgical, Inc.":                "Alcon Inc.",
    "Alcon Manufacturing, Ltd.":           "Alcon Inc.",
    "Alcon Laboratories Holding, Corp.":   "Alcon Inc.",
    "Alcon Universal, Ltd.":               "Alcon Inc.",

    "Eli Lilly And, Co.":                  "Eli Lilly and Co.",
    "Eli Lilly and, Co.":                  "Eli Lilly and Co.",
    "Eli Lilly &, Co.":                    "Eli Lilly and Co.",
    "Eli Lilly & Co., Inc.":               "Eli Lilly and Co.",
    "Eli Lilly & Company, Inc.":           "Eli Lilly and Co.",

    "Toyota Motor, Corp.":                 "Toyota Motor Corp.",
    "Toyota Motor North America, Inc.":    "Toyota Motor Corp.",
    "Toyota Motor Sales, U.S.A., Inc.":    "Toyota Motor Corp.",
    "Toyota Motor Sales USA, Inc.":        "Toyota Motor Corp.",
    "Toyota Motor Engineering & Manufacturing North America, Inc.": "Toyota Motor Corp.",
    "Toyota Motor Manufacturing, Kentucky, Inc.": "Toyota Motor Corp.",
    "Toyota Motor, Co.":                   "Toyota Motor Corp.",
    "Toyota Motors, Corp.":                "Toyota Motor Corp.",
    "Toyota Motor Manufacturing USA, Inc.": "Toyota Motor Corp.",

    "Volkswagen Group of America, Inc.":   "Volkswagen AG",
    "Volkswagen of America, Inc.":         "Volkswagen AG",
    "Volkswagen of America":               "Volkswagen AG",
    "Volkswagen Group Of America, Inc.":   "Volkswagen AG",
    "Volkswagen Of America, Inc.":         "Volkswagen AG",

    "Kia Motors America, Inc.":            "Kia Motors Corp.",
    "Kia Motors, Corp.":                   "Kia Motors Corp.",
    "Kia Motors Manufacturing Georgia, Inc.": "Kia Motors Corp.",
    "KIA Motors America, Inc.":            "Kia Motors Corp.",
    "Kia Motors Co, Ltd.":                 "Kia Motors Corp.",
    "KIA Motors, Corp.":                   "Kia Motors Corp.",
    "Kia Motors, Co.":                     "Kia Motors Corp.",

    "ASUS Computer International":         "ASUSTeK Computer Inc.",
    "Asustek Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asus Computer International":         "ASUSTeK Computer Inc.",
    "Asus Computer International, Inc.":   "ASUSTeK Computer Inc.",
    "ASUSTeK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "ASUSTek Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asus Computer Intl":                  "ASUSTeK Computer Inc.",
    "ASUS Computer International, Inc.":   "ASUSTeK Computer Inc.",
    "ASUS Computer International, Corp.":  "ASUSTeK Computer Inc.",
    "Asus Computer International, Corp.":  "ASUSTeK Computer Inc.",
    "ASUSteK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "ASUSTEK Computer, Inc.":              "ASUSTeK Computer Inc.",
    "Asustek Computer,Inc":                "ASUSTeK Computer Inc.",
    "ASUSTeK COMPUTER, Inc.":              "ASUSTeK Computer Inc.",

    "Lenovo (United States), Inc.":        "Lenovo Group Ltd.",
    "Lenovo Group, Ltd.":                  "Lenovo Group Ltd.",
    "Lenovo Holding Company, Inc.":        "Lenovo Group Ltd.",
    "Lenovo, Inc.":                        "Lenovo Group Ltd.",
    "Lenovo United States, Inc.":          "Lenovo Group Ltd.",
    "Lenovo (USA), Inc.":                  "Lenovo Group Ltd.",
    "Lenovo Group":                        "Lenovo Group Ltd.",
    "Lenovo International":                "Lenovo Group Ltd.",
    "Lenovo Group., Ltd.":                 "Lenovo Group Ltd.",

    "Sears, Roebuck and, Co.":             "Sears Holdings Corp.",
    "Sears Roebuck &, Co.":                "Sears Holdings Corp.",
    "Sears Brands, LLC":                   "Sears Holdings Corp.",
    "Sears Holdings Management, Corp.":    "Sears Holdings Corp.",
    "Sears Holding, Corp.":                "Sears Holdings Corp.",
    "Sears Roebuck And, Co.":              "Sears Holdings Corp.",

    "Time Warner Cable, Inc.":             "Time Warner Inc.",
    "Time Warner, Inc.":                   "Time Warner Inc.",
    "Time Warner Cable, LLC":              "Time Warner Inc.",
    "Time Warner Entertainment Company L P": "Time Warner Inc.",
    "AOL Time Warner, Inc.":               "Time Warner Inc.",

    "Warner-Lambert Company, LLC":         "Pfizer Inc.",
    "Warner-Lambert, Co.":                 "Pfizer Inc.",
    "Warner Lambert Company, LLC":         "Pfizer Inc.",
    "Warner Lambert, Co.":                 "Pfizer Inc.",

    "Par Pharmaceutical, Inc.":            "Par Pharmaceutical Inc.",
    "Par Pharmaceutical Companies, Inc.":  "Par Pharmaceutical Inc.",

    "Procter & Gamble, Co.":               "Procter & Gamble Co.",

    "Mitsubishi Electric, Corp.":          "Mitsubishi Electric Corp.",

    "Colgate-Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate-Palmolive Co., Inc.":         "Colgate-Palmolive Co.",
    "Colgate Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate Palmolive Co., Inc.":         "Colgate-Palmolive Co.",
    "Colgate/Palmolive, Co.":              "Colgate-Palmolive Co.",
    "Colgate-Palmolive Company, Inc.":     "Colgate-Palmolive Co.",

    "Smithkline Beecham, Corp.":           "GlaxoSmithKline PLC",
    "SmithKline Beecham, Corp.":           "GlaxoSmithKline PLC",
    "Smithkline Beecham Plc":              "GlaxoSmithKline PLC",
    "Smithkline Beecham, P.L.C.":          "GlaxoSmithKline PLC",
    "SmithKline Beecham (Cork), Ltd.":     "GlaxoSmithKline PLC",
    "Smithkline Beecham Consumer Healthcare, L.P.": "GlaxoSmithKline PLC",
    "Glaxosmithkline":                     "GlaxoSmithKline PLC",
    "Glaxosmithkline Plc":                 "GlaxoSmithKline PLC",
    "Glaxosmithkline Consumer Healthcare, L.P.": "GlaxoSmithKline PLC",
    "Glaxosmithkline, Corp.":              "GlaxoSmithKline PLC",
    "GLAXOSMITHKLINE p.l.c.":             "GlaxoSmithKline PLC",

    "Forest Laboratories Holdings, Ltd.":  "Forest Laboratories Inc.",
    "Forest Laboratories, Inc.":           "Forest Laboratories Inc.",
    "Forest Laboratories, LLC":            "Forest Laboratories Inc.",
    "Forest Laboratories Holding, Ltd.":   "Forest Laboratories Inc.",

    "Barr Laboratories, Inc.":             "Barr Laboratories Inc.",
    "Barr Laboratories, Inc.,":            "Barr Laboratories Inc.",
    "Barr Laboratories":                   "Barr Laboratories Inc.",

    "Hospira, Inc.":                       "Hospira Inc.",
    "Hospira Australia Pty, Ltd.":         "Hospira Inc.",
    "Hospira Worldwide, Inc.":             "Hospira Inc.",

    "Cordis, Corp.":                       "Cordis Corp.",
    "Cordis, LLC":                         "Cordis Corp.",
    "Cordis Endovascular Systems, Inc.":   "Cordis Corp.",

    "Gateway, Inc.":                       "Gateway Inc.",
    "Gateway Companies, Inc.":             "Gateway Inc.",
    "Gateway Country Stores, LLC":         "Gateway Inc.",
    "Gateway 2000, Inc.":                  "Gateway Inc.",

    "Freescale Semiconductor, Inc.":       "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Holdings I, Ltd.": "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Japan, Ltd.": "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Pte., Ltd.":  "Freescale Semiconductor Inc.",
    "Freescale Semiconductor Taiwan, Ltd.": "Freescale Semiconductor Inc.",

    "Endo Pharmaceuticals, Inc.":          "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Solutions, Inc.": "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Holding, Inc.":  "Endo Pharmaceuticals Inc.",
    "Endo Pharmaceuticals Holdings, Inc.": "Endo Pharmaceuticals Inc.",

    "Baxter Healthcare, Corp.":            "Baxter International Inc.",
    "Baxter International, Inc.":          "Baxter International Inc.",
    "Baxter Healthcare S.A.":              "Baxter International Inc.",
    "Baxter Diagnostics, Inc.":            "Baxter International Inc.",
    "Baxter Pharmaceutical Products, Inc.": "Baxter International Inc.",
    "Baxter Healthcare, Co.":              "Baxter International Inc.",

    "Black & Decker (U.S.), Inc.":         "Stanley Black & Decker Inc.",
    "Black & Decker, Inc.":                "Stanley Black & Decker Inc.",
    "Black & Decker, Corp.":               "Stanley Black & Decker Inc.",
    "Stanley Black & Decker, Inc.":        "Stanley Black & Decker Inc.",
    "Black & Decker (US), Inc.":           "Stanley Black & Decker Inc.",
    "The Black & Decker, Corp.":           "Stanley Black & Decker Inc.",
    "Black & Decker U.S., Inc.":           "Stanley Black & Decker Inc.",

    "Robert Bosch, LLC":                   "Robert Bosch GmbH",
    "Robert Bosch GmbH":                   "Robert Bosch GmbH",
    "Robert Bosch, Corp.":                 "Robert Bosch GmbH",
    "Robert Bosch Healthcare Systems, Inc.": "Robert Bosch GmbH",
    "Robert Bosch Tool, Corp.":            "Robert Bosch GmbH",
    "Robert Bosch North America, Corp.":   "Robert Bosch GmbH",
    "Robert Bosch Power Tool, Corp.":      "Robert Bosch GmbH",
    "Robert Bosch Gmbh":                   "Robert Bosch GmbH",
    "Bosch Security Systems, Inc.":        "Robert Bosch GmbH",

    "The Procter & Gamble, Co.":           "Procter & Gamble Co.",
    "Procter & Gamble Distributing, Co.":  "Procter & Gamble Co.",
    "Procter & Gamble Manufacturing, Co.": "Procter & Gamble Co.",
    "Procter and Gamble, Co.":             "Procter & Gamble Co.",

    "Allergan Sales, LLC":                 "Allergan Inc.",
    "Allergan USA, Inc.":                  "Allergan Inc.",
    "Allergan Sales, Inc.":                "Allergan Inc.",
    "Allergan Plc":                        "Allergan Inc.",
    "Allergan Optical, Inc.":              "Allergan Inc.",
    "Allergan Usa, Inc.":                  "Allergan Inc.",

    "Sandisk, Corp.":                      "SanDisk Corp.",
    "SanDisk, Corp.":                      "SanDisk Corp.",

    "Logitech, Inc.":                      "Logitech International S.A.",
    "Logitech International SA":           "Logitech International S.A.",
    "Logitech Europe S A":                 "Logitech International S.A.",
    "Logitech International S.A":          "Logitech International S.A.",
    "Logitech, LLC":                       "Logitech International S.A.",
    "Logitech Europe S.A.":                "Logitech International S.A.",

    "Garmin USA, Inc.":                    "Garmin Ltd.",
    "Garmin, Ltd.":                        "Garmin Ltd.",
    "Garmin, Corp.":                       "Garmin Ltd.",
    "Garmin International":                "Garmin Ltd.",
    "Garmin North America, Inc.":          "Garmin Ltd.",

    "Raytheon, Co.":                       "Raytheon Co.",
    "Raytheon Applied Signal Technology, Inc.": "Raytheon Co.",
    "Raytheon BBN Technologies, Corp.":    "Raytheon Co.",

    "Siemens, Corp.":                      "Siemens AG",
    "Siemens AG":                          "Siemens AG",
    "Siemens Medical Solutions USA, Inc.": "Siemens AG",
    "Siemens Enterprise Communications, Inc.": "Siemens AG",
    "Siemens Industry, Inc.":              "Siemens AG",
    "Siemens Medical Systems, Inc.":       "Siemens AG",
    "Siemens Energy & Automation, Inc.":   "Siemens AG",
    "Siemens Communications, Inc.":        "Siemens AG",

    "Koninklijke Philips Electronics NV":  "Koninklijke Philips N.V.",
    "U S Philips, Corp.":                  "Koninklijke Philips N.V.",
    "Koninklijke Philips Electronics N V": "Koninklijke Philips N.V.",

    "Cephalon, Inc.":                      "Teva Pharmaceutical Industries",
    "Cephalon France":                     "Teva Pharmaceutical Industries",

    "Genzyme Corp.":                       "Sanofi S.A.",
    "Genzyme Surgical Products, Corp.":    "Sanofi S.A.",

    "Ronald A Katz Technology Licensing L P":    "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A Katz Technology Licensing LP":     "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A Katz Technology Licensing, LP":    "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A Katz Technology":                  "Ronald A. Katz Technology Licensing L.P.",
    "Ronald A. Katz Technology Licensing, L.P.": "Ronald A. Katz Technology Licensing L.P.",

    "WI-Lan, Inc.":                        "WI-LAN Inc.",
    "WI-LAN, Inc.":                        "WI-LAN Inc.",
    "Wi-Lan, Inc.":                        "WI-LAN Inc.",
    "Wi-Lan USA, Inc.":                    "WI-LAN Inc.",
    "Wi-LAN, Inc.":                        "WI-LAN Inc.",
    "Wi-LAN USA, Inc.":                    "WI-LAN Inc.",
    "Wi-Lan Technologies, Corp.":          "WI-LAN Inc.",

    "Andrx, Corp.":                        "Actavis Inc.",
    "Andrx Pharmaceuticals, Inc.":         "Actavis Inc.",
    "Andrx Pharmaceuticals, LLC":          "Actavis Inc.",
    "Andrx Labs, LLC":                     "Actavis Inc.",
    "Andrx Corporation, Inc.":             "Actavis Inc.",

    "Barr Pharmaceuticals, Inc.":          "Teva Pharmaceutical Industries",
    "Barr Pharmaceuticals, LLC":           "Teva Pharmaceutical Industries",

    "ZTE Solutions, Inc.":                 "ZTE Corp.",
    "ZTE USA, Inc.":                       "ZTE Corp.",
    "ZTE (TX), Inc.":                      "ZTE Corp.",
    "ZTE (United States), Inc.":           "ZTE Corp.",

    "Sony Ericsson Mobile Communications AB":        "Sony Corp.",
    "Sony Ericsson Mobile Communications, Inc.":     "Sony Corp.",
    "Sony Ericsson Mobile Communications, AB":       "Sony Corp.",
    "Sony Ericsson Mobile Commications (USA), Inc.": "Sony Corp.",
    "Sony Ericsson Communications, (USA), Inc.":     "Sony Corp.",
    "Sony Ericsson Mobile Communication USA, Inc.":  "Sony Corp.",

    "JVC Americas, Corp.":                 "JVC Kenwood Corp.",
    "US JVC, Corp.":                       "JVC Kenwood Corp.",
    "JVC Kenwood, Corp.":                  "JVC Kenwood Corp.",
    "Jvc Americas, Corp.":                 "JVC Kenwood Corp.",
    "U.S. JVC, Corp.":                     "JVC Kenwood Corp.",
    "JVC America, Inc.":                   "JVC Kenwood Corp.",
    "JVC Company of America":              "JVC Kenwood Corp.",

    "SAP America, Inc.":                   "SAP AG",
    "Sap Ag":                              "SAP AG",
    "Sap America, Inc.":                   "SAP AG",
    "Sap, Ag":                             "SAP AG",

    "NEC Corporation of America":          "NEC Corp.",
    "NEC Corporation of America, Inc.":    "NEC Corp.",
    "NEC Corp. of America":                "NEC Corp.",
    "NEC Corp. of America, Inc.":          "NEC Corp.",

    "Mylan Laboratories, Ltd.":            "Mylan Inc.",

    "Kimberly-Clark Worldwide, Inc.":      "Kimberly-Clark Corp.",
    "Kimberly-Clark Global Sales, LLC":    "Kimberly-Clark Corp.",
    "Kimberly-Clark Global Sales, Inc.":   "Kimberly-Clark Corp.",
    "Kimberly-clark, Corp.":               "Kimberly-Clark Corp.",

    "Dell Computer, Corp.":                "Dell Inc.",
    "Dell Computer, Inc.":                 "Dell Inc.",
    "Dell Marketing, Corp.":               "Dell Inc.",
    "Dell, Inc.,":                         "Dell Inc.",

    "DataTreasury, Corp.":                 "Datatreasury Corp.",
    "Datatreasury, Corp.":                 "Datatreasury Corp.",

    "Sun Pharmaceutical Industries, Ltd.": "Sun Pharmaceutical Industries",
    "Sun Pharmaceutical Industries, Inc.": "Sun Pharmaceutical Industries",
    "Sun Pharma Global FZE":               "Sun Pharmaceutical Industries",
    "Sun Pharmaceuticals Industries, Inc.": "Sun Pharmaceutical Industries",
    "Sun Pharmaceuticals Industries, Ltd.": "Sun Pharmaceutical Industries",

    "Janssen Pharmaceuticals, Inc.":        "Johnson & Johnson",
    "Janssen Pharmaceutica N.V.":           "Johnson & Johnson",
    "Janssen Products, L.P.":               "Johnson & Johnson",
    "Janssen Biotech, Inc.":                "Johnson & Johnson",
    "Janssen Pharmaceutica Products, L.P.": "Johnson & Johnson",
    "Janssen, L.P.":                        "Johnson & Johnson",

    "J.C. Penney Corporation, Inc.":       "J.C. Penney Co.",
    "J.C. Penney Company, Inc.":           "J.C. Penney Co.",
    "J.C. Penney, Co.":                    "J.C. Penney Co.",
    "J.C. Penney CO., Inc.":               "J.C. Penney Co.",
    "J.C. Penney Co., Inc.":               "J.C. Penney Co.",

    "Lowe's Companies, Inc.":              "Lowe's Companies Inc.",
    "Lowe's Home Centers, Inc.":           "Lowe's Companies Inc.",
    "Lowe's HIW, Inc.":                    "Lowe's Companies Inc.",
    "Lowes Home Centers, Inc.":            "Lowe's Companies Inc.",
    "Lowe's Home Centers, LLC":            "Lowe's Companies Inc.",

    "American Honda Motor Co., Inc.":       "Honda Motor Co. Ltd.",
    "American Honda Motor Co, Inc.":        "Honda Motor Co. Ltd.",
    "Honda North America, Inc.":            "Honda Motor Co. Ltd.",
    "American Honda Motor Company, Inc.":   "Honda Motor Co. Ltd.",
    "Honda Manufacturing of Alabama, LLC":  "Honda Motor Co. Ltd.",
    "Honda of America Manufacturing, Inc.": "Honda Motor Co. Ltd.",
    "Honda Motor Co., Ltd.":                "Honda Motor Co. Ltd.",

    "Reebok International, Ltd.":          "Adidas AG",
    "Reebok Interanation, Ltd.":           "Adidas AG",

    "Scimed Life Systems, Inc.":           "Boston Scientific Corp.",
    "SciMed Life Systems, Inc.":           "Boston Scientific Corp.",
    "Boston Scientific SciMed, Inc.":      "Boston Scientific Corp.",

    "Warner Chilcott Company, LLC":        "Actavis Inc.",
    "Warner Chilcott (Us), LLC":           "Actavis Inc.",
    "Warner Chilcott (US), LLC":           "Actavis Inc.",
    "Warner Chilcott Company, Inc.":       "Actavis Inc.",
    "Warner Chilcott, Inc.":               "Actavis Inc.",
    "Warner Chilcott PLC":                 "Actavis Inc.",

    "Purdue Pharmaceuticals, L.P.":         "Purdue Pharma L.P.",
    "Purdue Pharma, L.P.":                  "Purdue Pharma L.P.",
    "Purdue Pharma Products LP":            "Purdue Pharma L.P.",
    "Purdue Pharmaceutical Products, L.P.": "Purdue Pharma L.P.",
    "The Purdue Pharma, Co.":               "Purdue Pharma L.P.",
    "Purdue Pharma LP":                     "Purdue Pharma L.P.",
    "Purdue Pharmaceuticals LP":            "Purdue Pharma L.P.",

    "AOL, Inc.":                           "AOL Inc.",
    "Aol, LLC":                            "AOL Inc.",
    "AOL Advertising, Inc.":               "AOL Inc.",
    "Aol, Inc.":                           "AOL Inc.",
    "Aol":                                 "AOL Inc.",

    "Hitachi America, Ltd.":                        "Hitachi Ltd.",
    "Hitachi Global Storage Technologies, Inc.":    "Hitachi Ltd.",
    "Hitachi Data Systems, Corp.":                  "Hitachi Ltd.",
    "Hitachi Home Electronics (America), Inc.":     "Hitachi Ltd.",
    "Hitachi Semiconductor (America), Inc.":        "Hitachi Ltd.",

    "Casio America, Inc.":                 "Casio Computer Co. Ltd.",
    "Casio Computer Co., Ltd.":            "Casio Computer Co. Ltd.",
    "Casio, Inc.":                         "Casio Computer Co. Ltd.",
    "Casio Computer Co, Ltd.":             "Casio Computer Co. Ltd.",
    "Casio Phonemate, Inc.":               "Casio Computer Co. Ltd.",

    "Nikon, Inc.":                         "Nikon Corp.",
    "Nikon, Corp.":                        "Nikon Corp.",
    "Nikon Precision, Inc.":               "Nikon Corp.",
    "Nikon Americas, Inc.":                "Nikon Corp.",
    "Nikon Instruments, Inc.":             "Nikon Corp.",

    "Brother International, Corp.":        "Brother Industries Ltd.",
    "Brother Industries, Ltd.":            "Brother Industries Ltd.",

    "Pantech Wireless, Inc.":              "Pantech Co. Ltd.",
    "Pantech Co., Ltd.":                   "Pantech Co. Ltd.",
    "Pantech & Curitel Communications, Inc.": "Pantech Co. Ltd.",
    "Pantech, Corp.":                      "Pantech Co. Ltd.",
    "Pantech Co, Ltd.":                    "Pantech Co. Ltd.",
    "Pantech Company, Ltd.":               "Pantech Co. Ltd.",

    "Becton Dickinson and, Co.":           "Becton Dickinson and Co.",

    "JPMorgan Chase &, Co.":               "JPMorgan Chase & Co.",

    "Bank of America, Corp.":              "Bank of America Corp.",

    "Callaway Golf, Co.":                  "Callaway Golf Co.",

    "Avery Dennison, Corp.":               "Avery Dennison Corp.",

    "Infineon Technologies North America, Corp.": "Infineon Technologies AG",

    "Halliburton Energy Services, Inc.":   "Halliburton Co.",

    "Seagate Technology, LLC":             "Seagate Technology",

    "Western Digital, Corp.":              "Western Digital Corp.",

    "Nuance Communications, Inc.":         "Microsoft Corp.",

    "Agere Systems, Inc.":                 "Broadcom Corp.",
    "LSI, Corp.":                          "Broadcom Corp.",

    "Life Technologies, Corp.":            "Thermo Fisher Scientific Inc.",

    "Hospira Inc.":                        "Pfizer Inc.",

}

def get_parent_org(name: str) -> str:
    """Return canonical parent org name, or the original if not in map."""
    return PARENT_MAP.get(name, name)

names['standardized_org_name'] = names['standardized_org_name'].apply(get_parent_org)

In [83]:
leftover_mask = names['standardized_org_name'].str.match(
    r'^a,?\s*(Corp\.|LLC|Co\.|L\.P\.)?$',
    case=False, na=False
)

names.loc[leftover_mask, 'standardized_org_name'] = 'Non-Specific Entity'

In [84]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
names['standardized_org_name'].value_counts().head(100).reset_index()

,standardized_org_name,count
0,Non-Specific Entity,41950
1,Samsung Electronics,1952
2,Actavis Inc.,1713
3,Sony Corp.,1501
4,Pfizer Inc.,1241
5,Teva Pharmaceutical Industries,1209
6,Mylan Inc.,1146
7,LG Electronics,1100
8,Apple Inc.,1084
9,formerly known as,973


In [87]:
# GICS Sector mapping for standardized_org_name values
# GICS Sectors:
#   Information Technology
#   Health Care
#   Communication Services
#   Consumer Discretionary
#   Consumer Staples
#   Financials
#   Industrials
#   Materials
#   Energy
#   Utilities
#   Real Estate
#   Unknown  (patent trolls, LLCs with no clear industry, etc.)

GICS_MAP = {

    "Samsung Electronics":              "Information Technology",
    "Apple Inc.":                       "Information Technology",
    "Microsoft Corp.":                  "Information Technology",
    "LG Electronics":                   "Information Technology",
    "HTC Corp.":                        "Information Technology",
    "Toshiba Corp.":                    "Information Technology",
    "HP Inc.":                          "Information Technology",
    "BlackBerry Ltd.":                  "Information Technology",
    "Dell Inc.":                        "Information Technology",
    "Panasonic Corp.":                  "Information Technology",
    "Melvino Technologies, Ltd.":       "Information Technology",
    "ASUSTeK Computer Inc.":            "Information Technology",
    "Fujitsu Ltd.":                     "Information Technology",
    "Acer Inc.":                        "Information Technology",
    "Lenovo Group Ltd.":                "Information Technology",
    "Cisco Systems Inc.":               "Information Technology",
    "Nintendo Co. Ltd.":                "Information Technology",
    "Intel Corp.":                      "Information Technology",
    "Sharp Corp.":                      "Information Technology",
    "IBM Corp.":                        "Information Technology",
    "Broadcom Corp.":                   "Information Technology",
    "Hitachi Ltd.":                     "Information Technology",
    "Ericsson AB":                      "Information Technology",
    "Qualcomm Inc.":                    "Information Technology",
    "Oracle Corp.":                     "Information Technology",
    "Ricoh Co. Ltd.":                   "Information Technology",
    "Texas Instruments Inc.":           "Information Technology",
    "Eastman Kodak Co.":                "Information Technology",
    "Gateway Inc.":                     "Information Technology",
    "Robert Bosch GmbH":                "Information Technology",
    "Kyocera Corp.":                    "Information Technology",
    "Adaptix, Inc.":                    "Information Technology",
    "Adobe Inc.":                       "Information Technology",
    "Symantec Corp.":                   "Information Technology",
    "Canon Inc.":                       "Information Technology",
    "Seiko Epson Corp.":                "Information Technology",
    "NEC Corp.":                        "Information Technology",
    "Siemens AG":                       "Information Technology",
    "Maxim Integrated Products, Inc.":  "Information Technology",
    "Casio Computer Co. Ltd.":          "Information Technology",
    "Palm Inc.":                        "Information Technology",
    "Freescale Semiconductor Inc.":     "Information Technology",
    "AMD Inc.":                         "Information Technology",
    "SanDisk Corp.":                    "Information Technology",
    "Infineon Technologies AG":         "Information Technology",
    "Micron Technology, Inc.":          "Information Technology",
    "JVC Kenwood Corp.":                "Information Technology",
    "Nikon Corp.":                      "Information Technology",
    "Analog Devices, Inc.":             "Information Technology",
    "Marvell Semiconductor, Inc.":      "Information Technology",
    "STMicroelectronics, Inc.":         "Information Technology",
    "D-Link Systems, Inc.":             "Information Technology",
    "EMC, Corp.":                       "Information Technology",
    "Seagate Technology":               "Information Technology",
    "Atmel, Corp.":                     "Information Technology",
    "Western Digital Corp.":            "Information Technology",
    "National Semiconductor, Corp.":    "Information Technology",
    "Netgear, Inc.":                    "Information Technology",
    "SAP AG":                           "Information Technology",
    "Xerox Corp.":                      "Information Technology",
    "NCR Corp.":                        "Information Technology",
    "Sanyo Electric Co.":               "Information Technology",
    "Pantech Co. Ltd.":                 "Information Technology",
    "Vizio, Inc.":                      "Information Technology",
    "Brother Industries Ltd.":          "Information Technology",
    "Imation, Corp.":                   "Information Technology",
    "Lexmark International, Inc.":      "Information Technology",
    "Pioneer Electronics (USA), Inc.":  "Information Technology",
    "Mitsubishi Electric Corp.":        "Information Technology",
    "Logitech International S.A.":      "Information Technology",
    "Applied Materials, Inc.":          "Information Technology",
    "Thermo Fisher Scientific Inc.":    "Information Technology",
    "Illumina, Inc.":                   "Information Technology",
    "Viewsonic, Corp.":                 "Information Technology",
    "Imation, Corp.":                   "Information Technology",
    "Symbol Technologies, Inc.":        "Information Technology",
    "Huawei Technologies":              "Information Technology",
    "ZTE Corp.":                        "Information Technology",
    "Directed Electronics, Inc.":       "Information Technology",
    "Belkin International, Inc.":       "Information Technology",
    "Novatel Wireless, Inc.":           "Information Technology",
    "Juniper Networks Inc.":            "Information Technology",
    "Netgear, Inc.":                    "Information Technology",
    "Valve, Corp.":                     "Information Technology",
    "McAfee, Inc.":                     "Information Technology",
    "Intuit, Inc.":                     "Information Technology",
    "Avaya, Inc.":                      "Information Technology",
    "Nuance Communications, Inc.":      "Information Technology",
    "Garmin Ltd.":                      "Information Technology",
    "Thomas & Betts, Corp.":            "Information Technology",

    "AT&T Inc.":                        "Communication Services",
    "Verizon Communications":           "Communication Services",
    "Nokia Corp.":                      "Communication Services",
    "Motorola Inc.":                    "Communication Services",
    "Sprint Corp.":                     "Communication Services",
    "Google Inc.":                      "Communication Services",
    "T-Mobile USA Inc.":                "Communication Services",
    "Comcast Corp.":                    "Communication Services",
    "Yahoo Inc.":                       "Communication Services",
    "Meta Platforms Inc.":              "Communication Services",
    "Time Warner Inc.":                 "Communication Services",
    "AOL Inc.":                         "Communication Services",
    "Charter Communications, Inc.":     "Communication Services",
    "Cox Communications, Inc.":         "Communication Services",
    "Zynga, Inc.":                      "Communication Services",
    "Electronic Arts, Inc.":            "Communication Services",
    "Netflix Inc.":                     "Communication Services",
    "Amazon.com Inc.":                  "Communication Services",
    "eBay Inc.":                        "Communication Services",
    "Overstock.com, Inc.":              "Communication Services",
    "Expedia, Inc.":                    "Communication Services",
    "Groupon, Inc.":                    "Communication Services",
    "QVC, Inc.":                        "Communication Services",
    "HSN, Inc.":                        "Communication Services",

    "Actavis Inc.":                     "Health Care",
    "Pfizer Inc.":                      "Health Care",
    "Teva Pharmaceutical Industries":   "Health Care",
    "Mylan Inc.":                       "Health Care",
    "Novartis AG":                      "Health Care",
    "Sanofi S.A.":                      "Health Care",
    "AstraZeneca PLC":                  "Health Care",
    "Apotex Inc.":                      "Health Care",
    "Lupin Ltd.":                       "Health Care",
    "Abbott Laboratories":              "Health Care",
    "GlaxoSmithKline PLC":              "Health Care",
    "Dr. Reddy's Laboratories":         "Health Care",
    "Sandoz Inc.":                      "Health Care",
    "Boston Scientific Corp.":          "Health Care",
    "Merck & Co.":                      "Health Care",
    "Ranbaxy Laboratories":             "Health Care",
    "Par Pharmaceutical Inc.":          "Health Care",
    "Medtronic Inc.":                   "Health Care",
    "Johnson & Johnson":                "Health Care",
    "Sun Pharmaceutical Industries":    "Health Care",
    "Allergan Inc.":                    "Health Care",
    "Roche Holding AG":                 "Health Care",
    "Bayer AG":                         "Health Care",
    "Purdue Pharma L.P.":               "Health Care",
    "Alcon Inc.":                       "Health Care",
    "Barr Laboratories Inc.":           "Health Care",
    "Forest Laboratories Inc.":         "Health Care",
    "Genentech Inc.":                   "Health Care",
    "Eli Lilly and Co.":                "Health Care",
    "Bristol-Myers Squibb Co.":         "Health Care",
    "Impax Laboratories, Inc.":         "Health Care",
    "Hospira Inc.":                     "Health Care",
    "UCB S.A.":                         "Health Care",
    "Baxter International Inc.":        "Health Care",
    "Cordis Corp.":                     "Health Care",
    "Endo Pharmaceuticals Inc.":        "Health Care",
    "Roxane Laboratories, Inc.":        "Health Care",
    "Amneal Pharmaceuticals, LLC":      "Health Care",
    "Stryker Corp.":                    "Health Care",
    "Celgene Corp.":                    "Health Care",
    "Ecolab Inc.":                      "Health Care",
    "Perrigo, Co.":                     "Health Care",
    "Breckenridge Pharmaceutical, Inc.": "Health Care",
    "Aurobindo Pharma, Ltd.":           "Health Care",
    "Wockhardt, Ltd.":                  "Health Care",
    "Mallinckrodt, Inc.":               "Health Care",
    "Otsuka Pharmaceutical Co., Ltd.":  "Health Care",
    "Smith & Nephew, Inc.":             "Health Care",
    "Anchen Pharmaceuticals, Inc.":     "Health Care",
    "Zimmer, Inc.":                     "Health Care",
    "AbbVie, Inc.":                     "Health Care",
    "Gilead Sciences, Inc.":            "Health Care",
    "Amgen, Inc.":                      "Health Care",
    "Becton Dickinson and Co.":         "Health Care",
    "Biomet, Inc.":                     "Health Care",
    "C.R. Bard, Inc.":                  "Health Care",
    "Supernus Pharmaceuticals, Inc.":   "Health Care",
    "Medicis Pharmaceutical, Corp.":    "Health Care",
    "Hetero Labs, Ltd.":                "Health Care",
    "Cadila Healthcare, Ltd.":          "Health Care",
    "Accord Healthcare, Inc.":          "Health Care",
    "Jazz Pharmaceuticals, Inc.":       "Health Care",
    "Advanced Cardiovascular Systems, Inc.": "Health Care",
    "Honeywell International Inc.":     "Health Care",
    "Bausch & Lomb, Inc.":              "Health Care",
    "Lemelson Medical, Education & Research Foundation LP": "Health Care",
    "The P.F. Laboratories, Inc.":      "Health Care",
    "Shire, LLC":                       "Health Care",

    "Sony Corp.":                       "Consumer Discretionary",
    "Walmart Inc.":                     "Consumer Discretionary",
    "Target Corp.":                     "Consumer Discretionary",
    "Best Buy Co.":                     "Consumer Discretionary",
    "Sears Holdings Corp.":             "Consumer Discretionary",
    "Toyota Motor Corp.":               "Consumer Discretionary",
    "Hyundai Motor Co.":                "Consumer Discretionary",
    "Nissan Motor Co.":                 "Consumer Discretionary",
    "Ford Motor Co.":                   "Consumer Discretionary",
    "General Motors Co.":               "Consumer Discretionary",
    "Honda Motor Co. Ltd.":             "Consumer Discretionary",
    "Mercedes-Benz Group":              "Consumer Discretionary",
    "BMW AG":                           "Consumer Discretionary",
    "Volkswagen AG":                    "Consumer Discretionary",
    "Kia Motors Corp.":                 "Consumer Discretionary",
    "Nike Inc.":                        "Consumer Discretionary",
    "Adidas AG":                        "Consumer Discretionary",
    "Oakley, Inc.":                     "Consumer Discretionary",
    "Mattel, Inc.":                     "Consumer Discretionary",
    "Lowe's Companies Inc.":            "Consumer Discretionary",
    "J.C. Penney Co.":                  "Consumer Discretionary",
    "Macy's, Inc.":                     "Consumer Discretionary",
    "Nordstrom, Inc.":                  "Consumer Discretionary",
    "Kohl's Department Stores, Inc.":   "Consumer Discretionary",
    "Toys \"R\" Us, Inc.":              "Consumer Discretionary",
    "Dick's Sporting Goods, Inc.":      "Consumer Discretionary",
    "Cabela's, Inc.":                   "Consumer Discretionary",
    "Barnes & Noble Inc.":              "Consumer Discretionary",
    "Bed Bath & Beyond, Inc.":          "Consumer Discretionary",
    "Newegg, Inc.":                     "Consumer Discretionary",
    "Sharper Image, Corp.":             "Consumer Discretionary",
    "Callaway Golf Co.":                "Consumer Discretionary",
    "Precor, Inc.":                     "Consumer Discretionary",
    "Conair, Corp.":                    "Consumer Discretionary",
    "Sunbeam Products, Inc.":           "Consumer Discretionary",
    "Whirlpool Corp.":                  "Consumer Discretionary",
    "Kmart, Corp.":                     "Consumer Discretionary",
    "Williams-Sonoma, Inc.":            "Consumer Discretionary",
    "Mag Instrument, Inc.":             "Consumer Discretionary",
    "Vizio, Inc.":                      "Consumer Discretionary",
    "Reebok International, Ltd.":       "Consumer Discretionary",

    "Procter & Gamble Co.":             "Consumer Staples",
    "Kimberly-Clark Corp.":             "Consumer Staples",
    "Monsanto Co.":                     "Consumer Staples",
    "Walgreens Co.":                    "Consumer Staples",
    "Costco Wholesale Corp.":           "Consumer Staples",
    "Colgate-Palmolive Co.":            "Consumer Staples",
    "3M Co.":                           "Consumer Staples",
    "Rite Aid Corp.":                   "Consumer Staples",
    "CVS Pharmacy, Inc.":               "Consumer Staples",
    "Staples Inc.":                     "Consumer Staples",
    "Office Depot Inc.":                "Consumer Staples",

    "General Electric Co.":             "Industrials",
    "Honeywell International Inc.":     "Industrials",
    "Illinois Tool Works Inc.":         "Industrials",
    "Emerson Electric Co.":             "Industrials",
    "Stanley Black & Decker Inc.":      "Industrials",
    "Garmin Ltd.":                      "Industrials",
    "Leggett & Platt, Inc.":            "Industrials",
    "Halliburton Co.":                  "Industrials",
    "Baker Hughes, Inc.":               "Industrials",
    "Pall, Corp.":                      "Industrials",
    "Leviton Manufacturing Co., Inc.":  "Industrials",
    "American Airlines, Inc.":          "Industrials",
    "Weatherford International, Inc.":  "Industrials",

    "American Express Co.":             "Financials",
    "JPMorgan Chase & Co.":             "Financials",
    "Bank of America Corp.":            "Financials",
    "Datatreasury Corp.":               "Financials",

    "Eastman Kodak Co.":                "Materials",

    "Non-Specific Entity":              "Unknown",
    "GeoTag Inc.":                      "Unknown",
    "TQP Development, LLC":             "Unknown",
    "ArrivalStar S.A.":                 "Unknown",
    "eDekka, LLC":                      "Unknown",
    "Blue Spike, LLC":                  "Unknown",
    "Ronald A. Katz Technology Licensing L.P.": "Unknown",
    "Parallel Networks, LLC":           "Unknown",
    "Uniloc USA, Inc.":                 "Unknown",
    "Uniloc Luxembourg S.A.":           "Unknown",
    "Eclipse IP, LLC":                  "Unknown",
    "Data Carriers, LLC":               "Unknown",
    "Unified Messaging Solutions, LLC": "Unknown",
    "Landmark Technology, LLC":         "Unknown",
    "Phoenix Licensing, LLC":           "Unknown",
    "LPL Licensing, LLC":               "Unknown",
    "Lodsys Group, LLC":                "Unknown",
    "Hawk Technology Systems, LLC":     "Unknown",
    "Thermolife International, LLC":    "Unknown",
    "Brandywine Communications Technologies, LLC": "Unknown",
    "Rates Technology, Inc.":           "Unknown",
    "WI-LAN Inc.":                      "Unknown",
    "Orion IP, LLC":                    "Unknown",
    "NovelPoint Tracking, LLC":         "Unknown",
    "Patent Group, LLC":                "Unknown",
    "Clear With Computers, LLC":        "Unknown",
    "Walker Digital, LLC":              "Unknown",
    "Acacia Media Technologies, Corp.": "Unknown",
    "CTP Innovations, LLC":             "Unknown",
    "EMG Technology, LLC":              "Unknown",
    "Wyncomm, LLC":                     "Unknown",
    "Webvention, LLC":                  "Unknown",
    "Automated Transactions, LLC":      "Unknown",
    "BSG Tech, LLC":                    "Unknown",
    "Affinity Labs of Texas, LLC":      "Unknown",
    "Beacon Navigation GmbH":           "Unknown",
    "Traffic Information, LLC":         "Unknown",
    "Network Signatures, Inc.":         "Unknown",
    "Penovia, LLC":                     "Unknown",
    "Genaville, LLC":                   "Unknown",
    "Technology Licensing, Corp.":      "Unknown",
    "Financial Systems Innovation, LLC": "Unknown",
    "Golden Bridge Technology, Inc.":   "Unknown",
    "Innovative Automation, LLC":       "Unknown",
    "CryptoPeak Solutions, LLC":        "Unknown",
    "LBS Innovations, LLC":             "Unknown",
    "Flashpoint Technology, Inc.":      "Unknown",
    "Chrimar Systems, Inc.":            "Unknown",
    "Chrimar Holding Company, LLC":     "Unknown",
    "Vantage Point Technology, Inc.":   "Unknown",
    "American Vehicular Sciences, LLC": "Unknown",
    "Intellectual Ventures II, LLC":    "Unknown",
    "Intellectual Ventures I, LLC":     "Unknown",
    "Wetro Lan, LLC":                   "Unknown",
    "Rothschild Connected Devices Innovations,LLC": "Unknown",
    "Pi-Net International, Inc.":       "Unknown",
    "Norman IP Holdings, LLC":          "Unknown",
    "Symbology Innovations, LLC":       "Unknown",
    "Protegrity, Corp.":                "Unknown",
    "Marshall Feature Recognition, LLC": "Unknown",
    "Soverain Software, LLC":           "Unknown",
    "Cygnus Telecommunications Technology, LLC": "Unknown",
    "UbiComm, LLC":                     "Unknown",
    "Innovative Wireless Solutions, LLC": "Unknown",
    "Location Services IP, LLC":        "Unknown",
    "Dynamic Hosting Company, LLC":     "Unknown",
    "Select Retrieval, LLC":            "Unknown",
    "Shipping and Transit, LLC":        "Unknown",
    "NovelPoint Security, LLC":         "Unknown",
    "Mosaid Technologies, Inc.":        "Unknown",
    "FastVDO, LLC":                     "Unknown",
    "Technology Properties Limited, LLC": "Unknown",
    "Sockeye Licensing TX, LLC":        "Unknown",
    "Innovative Display Technologies, LLC": "Unknown",
    "Rothschild Location Technologies, LLC": "Unknown",
    "WIAV Networks, LLC":               "Unknown",
    "Internet Machines, LLC":           "Unknown",
    "SmartPhone Technologies, LLC":     "Unknown",
    "SmartFit Solutions, LLC":          "Unknown",
    "IP Innovation, LLC":               "Unknown",
    "Multiplayer Network Innovations, LLC": "Unknown",
    "Cellular Communications Equipment, LLC": "Unknown",
    "InMotion Imagery Technologies, LLC": "Unknown",
    "CYVA Research Holdings, LLC":      "Unknown",
    "Innovatio IP Ventures, LLC":       "Unknown",
    "DietGoal Innovations, LLC":        "Unknown",
    "Lodsys Group, LLC":                "Unknown",
    "EON Corp. IP Holdings, LLC":       "Unknown",
    "Pragmatus Telecom, LLC":           "Unknown",
    "Azure Networks, LLC":              "Unknown",
    "Saxon Innovations, LLC":           "Unknown",
    "Rembrandt Technologies LP":        "Unknown",
    "Trading Technologies International, Inc.": "Unknown",
    "Klausner Technologies, Inc.":      "Unknown",
    "Bandspeed, Inc.":                  "Unknown",
}

def classify_by_keyword(name: str) -> str:
    if not isinstance(name, str):
        return "Unknown"
    n = name.lower()
    if re.search(r'pharma|pharmaceutical|therapeutics|biotech|bioscience|laborator|medical|health|clinic|drug|oncolog|surgical|diagnostics', n):
        return "Health Care"
    if re.search(r'bank|financ|insurance|capital|investment|securit|mortgage|credit|lending', n):
        return "Financials"
    if re.search(r'motor|automotive|vehicle|automob', n):
        return "Consumer Discretionary"
    if re.search(r'telecom|wireless|cellular|mobile|broadband|cable|satellite', n):
        return "Communication Services"
    if re.search(r'semiconductor|software|computer|tech|digital|electronic|network|cyber|data|systems|logic|micro|nano|cloud|ai\b|machine learning', n):
        return "Information Technology"
    if re.search(r'energy|oil|gas|petroleum|mining|chemical', n):
        return "Energy"
    if re.search(r'retail|store|shop|market|supermarket|grocer', n):
        return "Consumer Staples"
    if re.search(r'aerospace|defense|aviation|construction|engineering|manufacturing|industrial', n):
        return "Industrials"
    return "Unknown"

def get_gics_sector(name: str) -> str:
    """Return GICS sector for a standardized org name."""
    if name in GICS_MAP:
        return GICS_MAP[name]
    return classify_by_keyword(name)

names['gics_sector'] = names['standardized_org_name'].apply(get_gics_sector)

In [88]:
unknown_mask = names['gics_sector'] == 'Unknown'
print(names[unknown_mask]['standardized_org_name'].value_counts().head(50))

standardized_org_name
Non-Specific Entity                            41950
formerly known as                                973
a Delaware limited liability, Co.                624
Koninklijke Philips N.V.                         579
GeoTag Inc.                                      556
David Folsom                                     465
a California limited liability, Co.              457
Ronald A. Katz Technology Licensing L.P.         406
TQP Development, LLC                             400
ArrivalStar S.A.                                 371
eDekka, LLC                                      365
Blue Spike, LLC                                  325
individually                                     314
Data Carriers, LLC                               286
Unified Messaging Solutions, LLC                 269
Uniloc USA, Inc.                                 262
David Keyzer                                     258
Laughlin Products, Inc.                          252
WI-LAN Inc.             

In [86]:
names.head(20)

,case_row_id,case_number,party_row_count,party_type,name_row_count,name,party_type_original,case_number_clean,entity_type,standardized_org_name
0,1,0:79-cv-06704-JCP,1,Plaintiff,1,Burroghs Wellcome Co.,Plaintiff,0:1979-cv-06704,Organization,"Burroghs Wellcome, Co."
1,1,0:79-cv-06704-JCP,2,Defendant,2,Generix Drug Corp.,Defendant,0:1979-cv-06704,Organization,"Generix Drug, Corp."
2,3,0:83-cv-06860-JAG,3,Plaintiff,3,Kenneth R. Cornwall,Plaintiff,0:1983-cv-06860,Independent,Kenneth R. Cornwall
3,3,0:83-cv-06860-JAG,4,Defendant,4,"U. S. COnstruction Manufacturing, Inc.",Defendant,0:1983-cv-06860,Organization,"U. S. COnstruction Manufacturing, Inc."
4,4,0:84-cv-06456-KLR,5,Plaintiff,5,"Monte Carlo Hairpieces, Inc.",Plaintiff,0:1984-cv-06456,Organization,"Monte Carlo Hairpieces, Inc."
5,4,0:84-cv-06456-KLR,6,Plaintiff,6,James L. Waters,Plaintiff,0:1984-cv-06456,Independent,James L. Waters
6,4,0:84-cv-06456-KLR,7,Defendant,7,On-Rite Hairpiece Company,Defendant,0:1984-cv-06456,Organization,"On-Rite Hairpiece, Co."
7,4,0:84-cv-06456-KLR,8,Defendant,8,Andrew O. Wright,Defendant,0:1984-cv-06456,Independent,Andrew O. Wright
8,5,0:84-cv-06726-WMH,9,Plaintiff,9,"Monaco Del, Rocco A. Sr.",Plaintiff,0:1984-cv-06726,Independent,"Monaco Del, Rocco A. Sr."
9,5,0:84-cv-06726-WMH,10,Plaintiff,10,Frances Monaco Del,Plaintiff,0:1984-cv-06726,Independent,Frances Monaco Del


In [85]:
names.to_csv('standardized_org_names.csv', index=False)

In [22]:
org_mask = names['entity_type'] == 'Organization'
print(f"Rows matching 'Organization': {org_mask.sum()}")
print(names['entity_type'].value_counts().head(10))

Rows matching 'Organization': 423986
entity_type
Organization    423986
Independent      75296
Unknown          60450
Name: count, dtype: int64


In [14]:
print(len(names))

559732


In [15]:
# Entries ending in "as"
print("=== Ending in 'as' ===")
print(names[names['party_type'] == 'Independent']['name'].dropna()[
    names[names['party_type'] == 'Independent']['name'].dropna().str.strip().str.split().str[-1] == 'as'
].value_counts().head(30))

# Entries ending in "the"
print("\n=== Ending in 'the' ===")
print(names[names['party_type'] == 'Independent']['name'].dropna()[
    names[names['party_type'] == 'Independent']['name'].dropna().str.strip().str.split().str[-1].str.lower() == 'the'
].value_counts().head(30))

=== Ending in 'as' ===
Series([], Name: count, dtype: int64)

=== Ending in 'the' ===
Series([], Name: count, dtype: int64)


In [16]:
# Entries ending in "as"
print("=== Ending in 'trust' ===")
print(names[names['party_type'] == 'Independent']['name'].dropna()[
    names[names['party_type'] == 'Independent']['name'].dropna().str.strip().str.split().str[-1] == 'trust'
].value_counts().head(30))

# Entries ending in "the"
print("\n=== Ending in 'as' ===")
print(names[names['party_type'] == 'Independent']['name'].dropna()[
    names[names['party_type'] == 'Independent']['name'].dropna().str.strip().str.split().str[-1].str.lower() == 'as'
].value_counts().head(30))

=== Ending in 'trust' ===
Series([], Name: count, dtype: int64)

=== Ending in 'as' ===
Series([], Name: count, dtype: int64)


In [17]:
names[names['name'].str.strip().str.lower().str.endswith('also known as')].head(30)

,case_row_id,case_number,party_row_count,party_type,name_row_count,name,party_type_original,case_number_clean,entity_type
187,52,0:91-cv-06495-KMM,176,Defendant,188,also known as,Defendant,0:1991-cv-06495,Unknown
587,99,0:94-cv-06156-WDF,485,Plaintiff,588,also known as,Plaintiff,0:1994-cv-06156,Unknown
591,99,0:94-cv-06156-WDF,487,Defendant,592,also known as,Defendant,0:1994-cv-06156,Unknown
594,99,0:94-cv-06156-WDF,488,Counter Claimant,595,also known as,Counter Claimant,0:1994-cv-06156,Unknown
597,99,0:94-cv-06156-WDF,489,Counter Defendant,598,also known as,Counter Defendant,0:1994-cv-06156,Unknown
1675,260,0:97-cv-07426-AJ,1317,Defendant,1676,also known as,Defendant,0:1997-cv-07426,Unknown
1682,260,0:97-cv-07426-AJ,1321,Counter Claimant,1683,also known as,Counter Claimant,0:1997-cv-07426,Unknown
2362,355,0:99-cv-00573-JMR-FLN,1872,Defendant,2363,also known as,Defendant,0:1999-cv-00573,Unknown
2365,355,0:99-cv-00573-JMR-FLN,1873,Counter Claimant,2366,also known as,Counter Claimant,0:1999-cv-00573,Unknown
3113,605,0:01-cv-06195-BSS,2463,Defendant,3114,also known as,Defendant,0:2001-cv-06195,Unknown


In [18]:
independents = names[names['party_type'] == 'Independent']['name'].dropna()
as_entries = independents[independents.str.contains(r'\bAS\b|\bAs\b', regex=True)]
print(as_entries.value_counts().head(30))

Series([], Name: count, dtype: int64)


In [19]:
independents = names[names['party_type'] == 'Independent']['name'].dropna()

last_words = independents.str.strip().str.split().str[-1]
print(independents[last_words.index].value_counts().head(50))

last_two = independents.str.strip().str.split().str[-2:].str.join(' ')
print(independents[last_two.index].value_counts().head(50))

Series([], Name: count, dtype: int64)
Series([], Name: count, dtype: int64)
